# 04 Baselines And Component Checks

Reviewer-facing baseline/component evidence notebook for the revision package.


## Setup


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print(f'Drive mount skipped or already mounted: {exc}')

DRIVE_ROOT = Path('/content/drive/MyDrive/ECG-Ramba')
REPO_URL = 'https://github.com/BrianNguyen29/ECG-RAMBA.git'
BRANCH = os.environ.get('ECG_RAMBA_BRANCH', 'main')
REPO_DIR = Path('/content/ECG-RAMBA')
MIRROR_REVISION_ROOT = DRIVE_ROOT / 'revision_artifacts' / 'reports' / 'revision'
LOCAL_RUNTIME_ROOT = Path('/content/ecg_ramba_runtime')

os.environ['ECG_RAMBA_DRIVE_ROOT'] = str(DRIVE_ROOT)
os.environ.setdefault('ECG_RAMBA_LOCAL_ROOT', str(LOCAL_RUNTIME_ROOT))
os.environ.setdefault('ECG_RAMBA_EXTRACT_DIR', str(LOCAL_RUNTIME_ROOT / 'chapman'))
os.environ.setdefault('ECG_RAMBA_USE_CLEAN_CACHE', '0')
os.environ.setdefault('ECG_RAMBA_SAVE_CLEAN_CACHE', '0')

def _run_setup(cmd, cwd=None, check=True):
    print(f'$ {cmd}', flush=True)
    result = subprocess.run(
        cmd,
        shell=True,
        cwd=str(cwd) if cwd else None,
        check=False,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    if result.stdout:
        print(result.stdout.rstrip())
    if check and result.returncode:
        raise subprocess.CalledProcessError(result.returncode, cmd, output=result.stdout)
    return result

def _git_setup(cmd, check=True):
    return _run_setup(cmd, cwd=REPO_DIR, check=check)

def _git_current_commit():
    result = _git_setup('git rev-parse --short HEAD', check=False)
    return result.stdout.strip() if result.returncode == 0 and result.stdout else 'unknown'

def _pull_or_continue(branch):
    before = _git_current_commit()
    status_result = _git_setup('git status --porcelain', check=False)
    status = status_result.stdout.strip() if status_result.stdout else ''
    if status:
        print('Local repo has changes before pull:')
        print(status[:4000])
    result = _git_setup(f'git pull --ff-only --autostash origin {branch}', check=False)
    after = _git_current_commit()
    if result.returncode:
        print('')
        print('=' * 80)
        print('WARNING: git pull failed; continuing with the currently checked-out repo.')
        print('This avoids deleting Drive files while long-running artifacts may exist.')
        print(f'Current commit: {after} (before pull: {before})')
        print('To force a clean code checkout later, rename the Drive repo folder or use a fresh clone.')
        print('=' * 80)
        if os.environ.get('ECG_RAMBA_ALLOW_STALE_REPO', '0') != '1':
            raise RuntimeError('git pull failed; refusing to continue with stale code. Set ECG_RAMBA_ALLOW_STALE_REPO=1 only for debugging.')
    else:
        print(f'Git update OK: {before} -> {after}')

if (REPO_DIR / '.git').exists():
    os.chdir(REPO_DIR)
    fetch_result = _git_setup('git fetch origin', check=False)
    if fetch_result.returncode:
        print('WARNING: git fetch failed; continuing with the currently checked-out repo.')
    current_branch_result = _git_setup('git branch --show-current', check=False)
    current_branch = current_branch_result.stdout.strip() if current_branch_result.stdout else ''
    if current_branch != BRANCH:
        checkout_result = _git_setup(f'git checkout {BRANCH}', check=False)
        if checkout_result.returncode:
            print(f'WARNING: git checkout {BRANCH} failed; continuing on branch {current_branch or "<detached>"}')
        else:
            current_branch = BRANCH
    if fetch_result.returncode == 0:
        _pull_or_continue(BRANCH)
elif (REPO_DIR / 'configs' / 'config.py').exists():
    os.chdir(REPO_DIR)
    print('Repo directory exists but is not a git checkout; skipping git pull.')
elif Path.cwd().joinpath('configs', 'config.py').exists():
    REPO_DIR = Path.cwd()
    os.chdir(REPO_DIR)
    if (REPO_DIR / '.git').exists():
        fetch_result = _run_setup('git fetch origin', check=False)
        if fetch_result.returncode == 0:
            _pull_or_continue(BRANCH)
        else:
            print('WARNING: git fetch failed; continuing with the currently checked-out repo.')
else:
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
    _run_setup(f'git clone -b {BRANCH} {REPO_URL} {REPO_DIR}')
    os.chdir(REPO_DIR)

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
_run_setup('git status --short --branch', check=False)


REQUIRED_REVISION_ARTIFACTS_FOR_04 = [
    'predictions/oof_final_ema_predictions.npz',
    'manifests/oof_final_ema_freeze_manifest.json',
    'metrics/calibration_ci_oof_final_ema_predictions.json',
    'manifests/oof_final_ema_prediction_run_manifest.json',
    'tables/oof_final_ema_class_summary.csv',
]
HRV_ONLY_AVAILABLE_ARTIFACTS_FOR_04 = [
    'predictions/hrv_only_oof_predictions.npz',
    'metrics/hrv_only_baseline_summary.json',
    'tables/table_hrv_only_class_metrics.csv',
    'manifests/hrv_domain_analysis_manifest.json',
]
NOTEBOOK05_AVAILABLE_ARTIFACTS_FOR_04 = [
    'predictions/hrv_only_oof_predictions.npz',
    'predictions/hrv_domain_oof_predictions.npz',
    'metrics/hrv_only_baseline_summary.json',
    'metrics/hrv_domain_classifier_summary.json',
    'metrics/hrv_domain_summary.csv',
    'metrics/hrv_robustness_component_check_summary.json',
    'metrics/robustness_summary.csv',
    'tables/table_hrv_only_class_metrics.csv',
    'tables/table_hrv_domain_status.csv',
    'tables/table_hrv_domain_classifier_confusion.csv',
    'tables/table_hrv_domain_classifier_fold_summary.csv',
    'tables/table_robustness.csv',
    'manifests/hrv_domain_analysis_manifest.json',
    'manifests/hrv_robustness_input_contract.json',
    'manifests/hrv_robustness_plan_alignment.json',
]
MINIROCKET_AVAILABLE_ARTIFACTS_FOR_04 = [
    'predictions/minirocket_only_oof_predictions.npz',
    'metrics/minirocket_only_baseline_summary.json',
    'tables/table_minirocket_only_class_metrics.csv',
    'tables/table_minirocket_only_fold_summary.csv',
    'manifests/minirocket_only_baseline_manifest.json',
]
RESNET1D_CNN_AVAILABLE_ARTIFACTS_FOR_04 = [
    'predictions/resnet1d_cnn_oof_predictions.npz',
    'predictions/resnet1d_cnn_slice_predictions.npz',
    'predictions/folds/resnet1d_cnn_fold1_predictions.npz',
    'predictions/folds/resnet1d_cnn_fold2_predictions.npz',
    'predictions/folds/resnet1d_cnn_fold3_predictions.npz',
    'predictions/folds/resnet1d_cnn_fold4_predictions.npz',
    'predictions/folds/resnet1d_cnn_fold5_predictions.npz',
    'metrics/resnet1d_cnn_baseline_summary.json',
    'tables/table_resnet1d_cnn_class_metrics.csv',
    'tables/table_resnet1d_cnn_fold_summary.csv',
    'manifests/resnet1d_cnn_baseline_manifest.json',
]
RAW_MAMBA_AVAILABLE_ARTIFACTS_FOR_04 = [
    'predictions/raw_mamba_oof_predictions.npz',
    'predictions/raw_mamba_slice_predictions.npz',
    'predictions/folds/raw_mamba_fold1_predictions.npz',
    'predictions/folds/raw_mamba_fold2_predictions.npz',
    'predictions/folds/raw_mamba_fold3_predictions.npz',
    'predictions/folds/raw_mamba_fold4_predictions.npz',
    'predictions/folds/raw_mamba_fold5_predictions.npz',
    'metrics/raw_mamba_baseline_summary.json',
    'tables/table_raw_mamba_class_metrics.csv',
    'tables/table_raw_mamba_fold_summary.csv',
    'manifests/raw_mamba_baseline_manifest.json',
]
TRANSFORMER_ECG_AVAILABLE_ARTIFACTS_FOR_04 = [
    'predictions/transformer_ecg_oof_predictions.npz',
    'predictions/transformer_ecg_slice_predictions.npz',
    'predictions/folds/transformer_ecg_fold1_predictions.npz',
    'predictions/folds/transformer_ecg_fold2_predictions.npz',
    'predictions/folds/transformer_ecg_fold3_predictions.npz',
    'predictions/folds/transformer_ecg_fold4_predictions.npz',
    'predictions/folds/transformer_ecg_fold5_predictions.npz',
    'metrics/transformer_ecg_baseline_summary.json',
    'tables/table_transformer_ecg_class_metrics.csv',
    'tables/table_transformer_ecg_fold_summary.csv',
    'manifests/transformer_ecg_baseline_manifest.json',
]

HYBRID_MORPHOLOGY_AVAILABLE_ARTIFACTS_FOR_04 = [
    'predictions/hybrid_morphology_oof_predictions.npz',
    'metrics/hybrid_morphology_baseline_summary.json',
    'tables/table_hybrid_morphology_class_metrics.csv',
    'tables/table_hybrid_morphology_fold_summary.csv',
    'manifests/hybrid_morphology_baseline_manifest.json',
]
PAIRED_BASELINE_AVAILABLE_ARTIFACTS_FOR_04 = [
    'metrics/paired_full_vs_minirocket_comparison.json',
    'metrics/paired_full_vs_minirocket_bootstrap_samples.csv',
    'tables/table_paired_full_vs_minirocket.csv',
    'manifests/paired_full_vs_minirocket_manifest.json',
    'metrics/paired_full_vs_resnet_comparison.json',
    'metrics/paired_full_vs_resnet_bootstrap_samples.csv',
    'tables/table_paired_full_vs_resnet.csv',
    'manifests/paired_full_vs_resnet_manifest.json',
    'metrics/paired_full_vs_raw_mamba_comparison.json',
    'metrics/paired_full_vs_raw_mamba_bootstrap_samples.csv',
    'tables/table_paired_full_vs_raw_mamba.csv',
    'manifests/paired_full_vs_raw_mamba_manifest.json',
    'metrics/paired_full_vs_transformer_comparison.json',
    'metrics/paired_full_vs_transformer_bootstrap_samples.csv',
    'tables/table_paired_full_vs_transformer.csv',
    'manifests/paired_full_vs_transformer_manifest.json',
    'metrics/paired_full_vs_hybrid_morphology_comparison.json',
    'metrics/paired_full_vs_hybrid_morphology_bootstrap_samples.csv',
    'tables/table_paired_full_vs_hybrid_morphology.csv',
    'manifests/paired_full_vs_hybrid_morphology_manifest.json',
]
OPTIONAL_BASELINE_ARTIFACTS_FOR_04 = (
    NOTEBOOK05_AVAILABLE_ARTIFACTS_FOR_04
    + MINIROCKET_AVAILABLE_ARTIFACTS_FOR_04
    + RESNET1D_CNN_AVAILABLE_ARTIFACTS_FOR_04
    + RAW_MAMBA_AVAILABLE_ARTIFACTS_FOR_04
    + TRANSFORMER_ECG_AVAILABLE_ARTIFACTS_FOR_04
    + HYBRID_MORPHOLOGY_AVAILABLE_ARTIFACTS_FOR_04
    + PAIRED_BASELINE_AVAILABLE_ARTIFACTS_FOR_04
)


def _is_transport_endpoint_error(exc):
    return isinstance(exc, OSError) and (
        getattr(exc, 'errno', None) == 107
        or 'Transport endpoint is not connected' in str(exc)
    )


def _drive_transport_message(path):
    return (
        'Google Drive filesystem disconnected while reading artifact: ' + str(path) + '\n'
        'This is a Colab/Drive mount failure, not a metric/protocol failure. '
        'Reconnect/remount Drive or restart the runtime, rerun Notebook 04 from Setup, '
        'and keep ECG_RAMBA_FULL_MIRROR_RESTORE unset. If only required Notebook 02/03 '
        'contracts are needed, set ECG_RAMBA_RESTORE_OPTIONAL_BASELINE_ARTIFACTS=0 before Setup '
        'to skip baseline-cache restore.'
    )


def _sha256_local(path):
    import hashlib
    path = Path(path)
    digest = hashlib.sha256()
    try:
        with path.open('rb') as handle:
            for chunk in iter(lambda: handle.read(1024 * 1024), b''):
                digest.update(chunk)
    except OSError as exc:
        if _is_transport_endpoint_error(exc):
            raise RuntimeError(_drive_transport_message(path)) from exc
        raise
    return digest.hexdigest()


def _mirror_manifest_rows(mirror_root):
    manifest_path = Path(mirror_root) / 'manifests' / 'mirror_manifest.json'
    if not manifest_path.exists():
        return manifest_path, {}
    payload = json.loads(manifest_path.read_text(encoding='utf-8'))
    declared_root = Path(payload.get('mirror_root', mirror_root))
    rows = {}
    for row in payload.get('artifacts', []):
        relative = row.get('relative_path')
        if not relative and row.get('mirror'):
            mirror_path = Path(row['mirror'])
            try:
                relative = mirror_path.relative_to(declared_root).as_posix()
            except ValueError:
                relative = mirror_path.name
        if relative:
            rows[Path(relative).as_posix()] = row
    return manifest_path, rows


def restore_required_revision_artifacts(required_rel_paths):
    import shutil

    destination_root = REPO_DIR / 'reports' / 'revision'
    manifest_path, rows = _mirror_manifest_rows(MIRROR_REVISION_ROOT)
    restored, mirror_path_copied, reused, unresolved = [], [], [], []
    for rel in required_rel_paths:
        rel = Path(rel).as_posix()
        destination = destination_root / rel
        row = rows.get(rel)
        needs_copy = not destination.exists()
        if row and destination.exists() and _sha256_local(destination) != row.get('sha256'):
            needs_copy = True
        if not needs_copy:
            reused.append(rel)
            continue

        source = MIRROR_REVISION_ROOT / rel
        copied = False
        if row and source.exists():
            expected_size = int(row.get('size_bytes', -1))
            if expected_size >= 0 and source.stat().st_size != expected_size:
                raise RuntimeError(f'Mirror size mismatch for required artifact: {rel}')
            if _sha256_local(source) != row.get('sha256'):
                raise RuntimeError(f'Mirror checksum mismatch for required artifact: {rel}')
            destination.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source, destination)
            if _sha256_local(destination) != row.get('sha256'):
                raise RuntimeError(f'Checksum mismatch after restoring required artifact: {rel}')
            restored.append(rel)
            copied = True
        if not copied and source.exists():
            destination.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source, destination)
            mirror_path_copied.append(rel)
            copied = True
            print(f'WARNING: copied {rel} from mirror path without a manifest row; downstream freeze SHA checks still apply.')
        if not copied:
            unresolved.append(rel)

    print(
        'Required revision artifact restore: '
        f'restored={len(restored)} mirror_path={len(mirror_path_copied)} reused={len(reused)} unresolved={len(unresolved)}'
    )
    if restored:
        print('Restored required artifacts:')
        for rel in restored:
            print(' -', rel)
    if mirror_path_copied:
        print('Copied required artifacts from mirror path without manifest rows; contract SHA checks still apply:')
        for rel in mirror_path_copied:
            print(' -', rel)
    if unresolved:
        print('WARNING: required artifacts were not found in the verified mirror manifest:', manifest_path)
        for rel in unresolved:
            print(' -', rel)


def restore_available_revision_artifacts(optional_rel_paths):
    import shutil

    destination_root = REPO_DIR / 'reports' / 'revision'
    manifest_path, rows = _mirror_manifest_rows(MIRROR_REVISION_ROOT)
    restored, reused, unavailable = [], [], []
    for rel in optional_rel_paths:
        rel = Path(rel).as_posix()
        source = MIRROR_REVISION_ROOT / rel
        row = rows.get(rel)
        if row and source.exists():
            expected_size = int(row.get('size_bytes', -1))
            if expected_size >= 0 and source.stat().st_size != expected_size:
                raise RuntimeError(f'Mirror size mismatch for available artifact: {rel}')
            source_sha = row.get('sha256')
            if _sha256_local(source) != source_sha:
                raise RuntimeError(f'Mirror checksum mismatch for available artifact: {rel}')
        elif source.exists():
            source_sha = _sha256_local(source)
        else:
            unavailable.append(rel)
            continue

        destination = destination_root / rel
        if destination.exists() and _sha256_local(destination) == source_sha:
            reused.append(rel)
            continue
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, destination)
        if _sha256_local(destination) != source_sha:
            raise RuntimeError(f'Checksum mismatch after restoring available artifact: {rel}')
        restored.append(rel)

    print(
        'Available revision artifact restore: '
        f'restored={len(restored)} reused={len(reused)} unavailable={len(unavailable)}'
    )
    if restored:
        print('Restored available revision artifacts from mirror:')
        for rel in restored:
            print(' -', rel)


if MIRROR_REVISION_ROOT.exists():
    run_full_mirror_restore = os.environ.get('ECG_RAMBA_FULL_MIRROR_RESTORE', '0') == '1'
    if run_full_mirror_restore:
        restore_result = _run_setup(
            f'python -u scripts/revision/artifact_mirror.py restore --mirror-root "{MIRROR_REVISION_ROOT}" --replace-mismatched',
            cwd=REPO_DIR,
            check=False,
        )
        if restore_result.returncode:
            print('WARNING: full checksum-verified mirror restore failed; trying selective restore for required inputs only.')
    else:
        print('Skipping full mirror restore in Notebook 04 setup. Set ECG_RAMBA_FULL_MIRROR_RESTORE=1 only if a full local mirror copy is required.')
    restore_required_revision_artifacts(REQUIRED_REVISION_ARTIFACTS_FOR_04)
    restore_optional_baseline_artifacts = os.environ.get('ECG_RAMBA_RESTORE_OPTIONAL_BASELINE_ARTIFACTS', '1') == '1'
    if restore_optional_baseline_artifacts:
        try:
            restore_available_revision_artifacts(OPTIONAL_BASELINE_ARTIFACTS_FOR_04)
        except RuntimeError as exc:
            if 'Google Drive filesystem disconnected' in str(exc):
                print('Required Notebook 02/03 artifacts were restored before optional baseline artifact restore failed.')
            raise
    else:
        print('Skipping optional baseline artifact restore in Notebook 04 setup because ECG_RAMBA_RESTORE_OPTIONAL_BASELINE_ARTIFACTS=0.')
else:
    print('Mirror not found; Notebook 04 will require Notebook 02/03 artifacts in the active repo:', MIRROR_REVISION_ROOT)

print('cwd       :', Path.cwd())
print('drive_root:', DRIVE_ROOT)
print('repo_dir  :', REPO_DIR)
print('branch    :', BRANCH)

cache_status = {
    'clean_raw_cache': DRIVE_ROOT / 'ecg_data_27c_subject.npz',
    'raw_minirocket_cache': DRIVE_ROOT / 'rocket_raw_N44186_C12_L5000_K10000_S42.npz',
    'hrv36_cache': DRIVE_ROOT / 'hrv36_N44186_C12_L5000.npz',
    'fold_pca_cache_dir': DRIVE_ROOT / 'revision_feature_cache',
}
print('cache policy:')
print('  ECG_RAMBA_USE_CLEAN_CACHE =', os.environ.get('ECG_RAMBA_USE_CLEAN_CACHE'))
print('  ECG_RAMBA_SAVE_CLEAN_CACHE=', os.environ.get('ECG_RAMBA_SAVE_CLEAN_CACHE'))
print('  ECG_RAMBA_EXTRACT_DIR     =', os.environ.get('ECG_RAMBA_EXTRACT_DIR'))
print('cache status:')
for name, path in cache_status.items():
    if path.is_dir():
        count = len(list(path.glob('*.npz')))
        print(f'  {name}: exists=True npz_count={count} path={path}')
    else:
        print(f'  {name}: exists={path.exists()} size={path.stat().st_size if path.exists() else None} path={path}')
fingerprinted_rocket_caches = sorted(DRIVE_ROOT.glob('rocket_raw_N44186_C12_L5000_K10000_S42_R*.npz'))
print('  raw_minirocket_record_fingerprinted_count:', len(fingerprinted_rocket_caches))
for path in fingerprinted_rocket_caches[:5]:
    print(f'    - {path.name} size={path.stat().st_size}')

def run(cmd, check=True, log_path=None, tail_lines=160):
    import time

    print(f'$ {cmd}', flush=True)
    if log_path is None:
        log_dir = Path('reports/revision/logs')
        log_dir.mkdir(parents=True, exist_ok=True)
        stamp = time.strftime('%Y%m%d_%H%M%S')
        log_path = log_dir / f'notebook_command_{stamp}.log'
    else:
        log_path = Path(log_path)
        log_path.parent.mkdir(parents=True, exist_ok=True)

    with log_path.open('w', encoding='utf-8') as log_file:
        proc = subprocess.Popen(
            cmd,
            shell=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            encoding='utf-8',
            errors='replace',
            bufsize=1,
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end='')
            log_file.write(line)
            log_file.flush()
        return_code = proc.wait()

    print(f'Command log: {log_path}')
    if check and return_code:
        lines = log_path.read_text(encoding='utf-8', errors='replace').splitlines()
        print(f'Command failed with exit code {return_code}. Last {min(tail_lines, len(lines))} log lines:')
        for line in lines[-tail_lines:]:
            print(line)
        raise subprocess.CalledProcessError(return_code, cmd)
    return subprocess.CompletedProcess(cmd, return_code)


def run_script_if_exists(script_path, command):
    path = Path(script_path)
    if path.exists():
        run(command)
    else:
        print(f'SKIP: {script_path} is not implemented yet.')
        print(f'Planned command: {command}')


## Scope And Contracts

This notebook is intentionally CPU-friendly and artifact-first. It validates the frozen OOF and calibration outputs, writes reusable evidence tables, and records which fair baselines are complete or still blocked. It does not run full ECG-RAMBA training. It can optionally fit the lightweight MiniRocket-only classifier under the frozen OOF protocol, while unavailable model-specific runners remain blocked.


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import math
import os
import shutil
import sys

import numpy as np
import pandas as pd
from IPython.display import display

if not Path.cwd().joinpath('configs', 'config.py').exists() and Path('/content/ECG-RAMBA/configs/config.py').exists():
    os.chdir('/content/ECG-RAMBA')
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

from scripts.revision.common import calibration_summary, git_commit, multilabel_metrics, save_csv, save_json, sha256_file

THRESHOLD = 0.5
N_BINS = 15
RUN_HEAVY_BASELINES = False

revision_root = Path('reports/revision')
OOF_STEM = 'oof_final_ema'
pred_path = revision_root / 'predictions' / f'{OOF_STEM}_predictions.npz'
freeze_path = revision_root / 'manifests' / f'{OOF_STEM}_freeze_manifest.json'
calibration_path = revision_root / 'metrics' / f'calibration_ci_{OOF_STEM}_predictions.json'
run_manifest_path = revision_root / 'manifests' / f'{OOF_STEM}_prediction_run_manifest.json'
class_table_path = revision_root / 'tables' / f'{OOF_STEM}_class_summary.csv'

required_inputs = {
    'frozen_oof_predictions': pred_path,
    'oof_final_ema_freeze_manifest': freeze_path,
    'calibration_ci': calibration_path,
    'oof_run_manifest': run_manifest_path,
    'oof_class_summary': class_table_path,
}
for name, path in required_inputs.items():
    print(f'{name:24s}: exists={path.exists()} size={path.stat().st_size if path.exists() else None} path={path}')
missing = [str(path) for path in required_inputs.values() if not path.exists()]
if missing and 'restore_required_revision_artifacts' in globals():
    print('Required inputs missing before contract validation; retrying selective artifact restore...')
    restore_required_revision_artifacts(REQUIRED_REVISION_ARTIFACTS_FOR_04)
    missing = [str(path) for path in required_inputs.values() if not path.exists()]
if missing:
    print('Required inputs still missing; attempting direct fallback restore from Drive roots.')
    drive_root = globals().get('DRIVE_ROOT', Path('/content/drive/MyDrive/ECG-Ramba'))
    mirror_root = drive_root / 'revision_artifacts' / 'reports' / 'revision'
    drive_repo_root = drive_root / 'ECG-RAMBA' / 'reports' / 'revision'
    fallback_roots = [mirror_root, drive_repo_root]
    restored_direct, unresolved_direct = [], []
    for destination in required_inputs.values():
        if destination.exists():
            continue
        rel = destination.relative_to(revision_root)
        source = next((root / rel for root in fallback_roots if (root / rel).exists()), None)
        if source is None:
            unresolved_direct.append(str(destination))
            continue
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, destination)
        restored_direct.append(f'{source} -> {destination}')
    print(f'Direct fallback restore: restored={len(restored_direct)} unresolved={len(unresolved_direct)}')
    for item in restored_direct:
        print(' - restored', item)
    for item in unresolved_direct:
        print(' - unresolved', item)
    missing = [str(path) for path in required_inputs.values() if not path.exists()]
if missing:
    raise FileNotFoundError(
        'Notebook 02/03 artifacts are required before notebook 04: ' + '; '.join(missing) +
        '. Run Notebook 04 from Setup, or verify the files exist under '
        '/content/drive/MyDrive/ECG-Ramba/revision_artifacts/reports/revision.'
    )

freeze = json.loads(freeze_path.read_text(encoding='utf-8'))
calibration = json.loads(calibration_path.read_text(encoding='utf-8'))
if freeze.get('status') != 'frozen' or freeze.get('manuscript_ready') is not True:
    raise ValueError('final_ema OOF freeze manifest must be status=frozen and manuscript_ready=true before baseline/component checks.')
if freeze.get('checkpoint_kind') != 'final_ema':
    raise ValueError('Notebook 04 requires checkpoint_kind=final_ema in the freeze manifest.')

frozen_artifacts = {row['path']: row for row in freeze.get('artifacts', [])}

def verify_frozen_file(path, label):
    rel = path.as_posix()
    if rel not in frozen_artifacts:
        raise ValueError(f'Freeze manifest does not include {rel}')
    current_sha = sha256_file(path)
    if current_sha != frozen_artifacts[rel]['sha256']:
        raise RuntimeError(f'{label} SHA256 changed after OOF freeze: {rel}')
    return current_sha

pred_sha = verify_frozen_file(pred_path, 'Prediction')
run_manifest_sha = verify_frozen_file(run_manifest_path, 'Run manifest')
class_table_sha = verify_frozen_file(class_table_path, 'Class table')
freeze_sha = sha256_file(freeze_path)
if calibration.get('predictions_sha256') != pred_sha:
    raise RuntimeError(
        'Calibration JSON does not match frozen OOF prediction SHA256. '
        f"json={calibration.get('predictions_sha256')} current={pred_sha}. "
        'Rerun Notebook 03 with FORCE_RERUN_CALIBRATION_CI=True, or use the updated Notebook 03 stale-artifact check.'
    )
if calibration.get('freeze_manifest_sha256') != freeze_sha:
    raise RuntimeError(
        'Calibration JSON does not match final_ema freeze manifest SHA256. '
        f"json={calibration.get('freeze_manifest_sha256')} current={freeze_sha}. "
        'This usually means Notebook 02 refreshed the freeze manifest after calibration was computed. '
        'Rerun Notebook 03 from Setup with FORCE_RERUN_CALIBRATION_CI=True, then publish/restore the mirror and rerun this cell.'
    )
expected_shape = {'y_prob': [freeze.get('validated_records'), freeze.get('n_classes')], 'y_true': [freeze.get('validated_records'), freeze.get('n_classes')]}
if calibration.get('shape') != expected_shape:
    raise RuntimeError(f'Calibration shape does not match freeze manifest: {calibration.get("shape")} != {expected_shape}')

input_contract = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'git_commit': git_commit(),
    'threshold': THRESHOLD,
    'n_bins': N_BINS,
    'inputs': {
        name: {'path': str(path), 'sha256': sha256_file(path), 'size_bytes': path.stat().st_size}
        for name, path in required_inputs.items()
    },
    'freeze_status': freeze.get('status'),
    'freeze_manuscript_ready': freeze.get('manuscript_ready'),
    'run_manifest_sha256': run_manifest_sha,
    'class_table_sha256': class_table_sha,
    'freeze_manifest_sha256': freeze_sha,
    'calibration_dataset': calibration.get('dataset'),
    'calibration_protocol': calibration.get('protocol'),
    'calibration_shape': calibration.get('shape'),
}
save_json(revision_root / 'manifests' / 'baseline_component_input_contract.json', input_contract)
print('Input contract validated and written:', revision_root / 'manifests' / 'baseline_component_input_contract.json')


## Revision Plan Alignment

The cells below bind notebook 04 to the tracked reviewer plan. They make incomplete fair baselines explicit instead of silently treating placeholders as evidence.


In [ ]:
registry = pd.read_csv('docs/revision_plan/experiment_registry.csv')
claims = pd.read_csv('docs/revision_plan/claim_evidence_map.csv')
tasks = pd.read_csv('docs/revision_plan/task_board.csv')

relevant_experiments = registry[registry['experiment_id'].isin(['EXP_BASE', 'EXP_CAL', 'EXP_POOL'])]
relevant_claims = claims[claims['claim_id'].isin(['C01', 'C02', 'C05', 'C06'])]
relevant_tasks = tasks[tasks['id'].isin(['A2', 'A3', 'A4', 'B1'])]

display(relevant_experiments)
display(relevant_claims)
display(relevant_tasks)

plan_alignment = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'git_commit': git_commit(),
    'experiments': relevant_experiments.to_dict(orient='records'),
    'claims': relevant_claims.to_dict(orient='records'),
    'tasks': relevant_tasks.to_dict(orient='records'),
}
save_json(revision_root / 'manifests' / 'baseline_component_plan_alignment.json', plan_alignment)
print('Wrote:', revision_root / 'manifests' / 'baseline_component_plan_alignment.json')


## No-Skill Reference Baselines

These are sanity references computed from the frozen OOF labels only. They are not substitutes for the fair model baselines requested by reviewers, but they help interpret whether the frozen model beats trivial prevalence/constant references under the same metric functions.


In [ ]:
with np.load(pred_path, allow_pickle=False) as data:
    y_true = np.asarray(data['y_true'], dtype=np.float32)
    y_prob_full = np.asarray(data['y_prob'], dtype=np.float32)
    class_names = data['class_names'].astype(str).tolist() if 'class_names' in data.files else list(range(y_true.shape[1]))

prevalence = y_true.mean(axis=0).astype(np.float32)
reference_predictions = {
    'full_ecg_ramba_frozen_oof': y_prob_full,
    'prevalence_probability_reference': np.tile(prevalence, (len(y_true), 1)).astype(np.float32),
    'zero_probability_reference': np.zeros_like(y_true, dtype=np.float32),
    'constant_0_5_probability_reference': np.full_like(y_true, 0.5, dtype=np.float32),
}

rows = []
for name, y_prob in reference_predictions.items():
    metrics = multilabel_metrics(y_true, y_prob, threshold=THRESHOLD)
    cal = calibration_summary(y_true, y_prob, n_bins=N_BINS)
    rows.append({
        'model_or_reference': name,
        'dataset': 'chapman_oof',
        'n_records': int(y_true.shape[0]),
        'n_classes': int(y_true.shape[1]),
        'threshold': THRESHOLD,
        'evidence_role': 'frozen_model' if name == 'full_ecg_ramba_frozen_oof' else 'no_skill_reference_not_fair_baseline',
        'uses_evaluation_labels_to_set_probability': name == 'prevalence_probability_reference',
        'fair_baseline_claim_eligible': False,
        **metrics,
        **cal,
    })

reference_df = pd.DataFrame(rows)
reference_csv = revision_root / 'metrics' / 'no_skill_reference_baselines.csv'
reference_table = revision_root / 'tables' / 'table_no_skill_reference_baselines.csv'
reference_df.to_csv(reference_csv, index=False)
reference_df.to_csv(reference_table, index=False)
print('Wrote:', reference_csv)
print('Wrote:', reference_table)
display(reference_df)


## MiniRocket-only Baseline Runner

Optional fold-safe MiniRocket-only baseline. Set `RUN_MINIROCKET_ONLY_BASELINE=True` for the first run; later runs can reuse the mirrored artifact.


In [ ]:
RUN_MINIROCKET_ONLY_BASELINE = False

minirocket_command = (
    'python -u scripts/revision/10_minirocket_only_baseline.py '
    '--oof-predictions reports/revision/predictions/oof_final_ema_predictions.npz '
    '--freeze-manifest reports/revision/manifests/oof_final_ema_freeze_manifest.json '
    '--expected-checkpoint-kind final_ema '
    '--threshold 0.5 --n-bins 15 --n-boot 1000 --reuse-predictions --backend torch_linear --torch-epochs 20 --batch-size 4096 --stats-batch-size 1024 --lr 1e-3 --weight-decay 1e-4 --standardize train_fold --device cuda'
)
print('MiniRocket-only can now be run as a fold-safe feature baseline. It may take time because it fits fold-safe standardized linear heads on 20,000 cached features in A100 GPU batches.')
print('MiniRocket-only: implemented=', Path('scripts/revision/10_minirocket_only_baseline.py').exists(), 'planned_command=', minirocket_command)

if RUN_MINIROCKET_ONLY_BASELINE:
    run(minirocket_command, log_path='reports/revision/logs/baseline_minirocket_only.log')
    if 'MIRROR_REVISION_ROOT' in globals():
        run(
            f'python -u scripts/revision/artifact_mirror.py publish --mirror-root "{MIRROR_REVISION_ROOT}"',
            log_path='reports/revision/logs/baseline_minirocket_only_immediate_mirror_publish.log',
        )
    else:
        print('Immediate mirror publish skipped because MIRROR_REVISION_ROOT is not defined.')
else:
    print('MiniRocket-only execution disabled. Set RUN_MINIROCKET_ONLY_BASELINE=True in this cell to run it before the Fair Baseline Completion Matrix.')


## ResNet1D/CNN Baseline Runner

Optional fold-safe raw-ECG architecture baseline. Set `RUN_RESNET1D_CNN_BASELINE=True` for the first run. Later runs reuse final or per-fold prediction artifacts, so interrupted Colab sessions can resume without restarting completed folds. For robustness-vs-ResNet stress inference, checkpoints must be created during training; set `RESNET_SAVE_CHECKPOINTS=True` and `RESNET_FORCE_RERUN=True` because existing prediction caches cannot reconstruct model weights.


In [ ]:
RUN_RESNET1D_CNN_BASELINE = 'auto'
RESNET_EPOCHS = 20
RESNET_BATCH_SIZE = 512
RESNET_NUM_WORKERS = 2
RESNET_N_BOOT = 1000
RESNET_LIMIT_RECORDS = 0
RESNET_FORCE_RERUN = 'auto'
RESNET_SAVE_CHECKPOINTS = True
RESNET_RETRAIN_FOR_MISSING_CHECKPOINTS = False  # Keep False to reuse completed baseline caches; True only for robustness checkpoint export.
RESNET_BASELINE_RELPATHS = [
    Path('predictions/resnet1d_cnn_oof_predictions.npz'),
    Path('predictions/resnet1d_cnn_slice_predictions.npz'),
    Path('metrics/resnet1d_cnn_baseline_summary.json'),
    Path('tables/table_resnet1d_cnn_class_metrics.csv'),
    Path('tables/table_resnet1d_cnn_fold_summary.csv'),
    Path('manifests/resnet1d_cnn_baseline_manifest.json'),
    *[Path(f'predictions/folds/resnet1d_cnn_fold{fold}_predictions.npz') for fold in range(1, 6)],
]
RESNET_CHECKPOINT_RELPATHS = [
    Path(f'experimental/resnet1d_cnn_checkpoints/fold{fold}_resnet1d_cnn_final.pt')
    for fold in range(1, 6)
]
missing_resnet_baseline_artifacts = [
    str(rel) for rel in RESNET_BASELINE_RELPATHS
    if not (revision_root / rel).exists() or (revision_root / rel).stat().st_size == 0
]
missing_resnet_checkpoint_folds = [
    fold for fold, rel in enumerate(RESNET_CHECKPOINT_RELPATHS, start=1)
    if not (revision_root / rel).exists() or (revision_root / rel).stat().st_size == 0
]
if RESNET_FORCE_RERUN == 'auto':
    RESNET_FORCE_RERUN = bool(missing_resnet_baseline_artifacts) or (
        bool(missing_resnet_checkpoint_folds) and bool(RESNET_RETRAIN_FOR_MISSING_CHECKPOINTS)
    )
print('Resolved RESNET_FORCE_RERUN:', RESNET_FORCE_RERUN)
print('Missing ResNet baseline artifacts:', missing_resnet_baseline_artifacts or 'none')
print('Missing ResNet checkpoint folds:', missing_resnet_checkpoint_folds or 'none')
if missing_resnet_checkpoint_folds and not RESNET_RETRAIN_FOR_MISSING_CHECKPOINTS:
    print('ResNet checkpoints are missing, but baseline outputs are reusable. Set RESNET_RETRAIN_FOR_MISSING_CHECKPOINTS=True only if Notebook 05 learned-comparator stress needs checkpoints.')

resnet_command = (
    'python -u scripts/revision/14_resnet1d_cnn_baseline.py '
    '--oof-predictions reports/revision/predictions/oof_final_ema_predictions.npz '
    '--freeze-manifest reports/revision/manifests/oof_final_ema_freeze_manifest.json '
    '--expected-checkpoint-kind final_ema '
    f'--epochs {RESNET_EPOCHS} --batch-size {RESNET_BATCH_SIZE} --num-workers {RESNET_NUM_WORKERS} '
    '--threshold 0.5 --n-bins 15 '
    f'--n-boot {RESNET_N_BOOT} '
    '--lr 1e-3 --weight-decay 1e-4 --dropout 0.20 --base-channels 64 '
    '--device cuda --amp --allow-tf32 --reuse-predictions'
)
if RESNET_FORCE_RERUN:
    resnet_command += ' --force-rerun'
if RESNET_SAVE_CHECKPOINTS:
    resnet_command += ' --save-checkpoints --checkpoint-dir reports/revision/experimental/resnet1d_cnn_checkpoints'

if RESNET_LIMIT_RECORDS:
    resnet_command += f' --limit-records {RESNET_LIMIT_RECORDS}'

resnet_artifact_relpaths = [
    *RESNET_BASELINE_RELPATHS,
    *RESNET_CHECKPOINT_RELPATHS,
]

def _print_resnet_artifact_status(root, label):
    root = Path(root)
    print()
    print(f'ResNet artifact status: {label} -> {root}')
    for rel in resnet_artifact_relpaths:
        path = root / rel
        print(f'  {rel}: exists={path.exists()} size={path.stat().st_size if path.exists() else None}')

print('ResNet1D/CNN baseline runner is fold-safe and raw-ECG-only. It does not use MiniRocket, HRV, PCA, or ECG-RAMBA checkpoints.')
print('Completed folds are cached under reports/revision/predictions/folds/ and reused on rerun if present in the active repo.')
print('Checkpoint export for robustness requires RESNET_RETRAIN_FOR_MISSING_CHECKPOINTS=True when checkpoints are absent; otherwise completed prediction caches are reused.')
print('After a successful run this cell immediately publishes reports/revision to the Drive mirror to survive Colab disconnects.')
print('ResNet1D/CNN: implemented=', Path('scripts/revision/14_resnet1d_cnn_baseline.py').exists(), 'planned_command=', resnet_command)
_print_resnet_artifact_status(revision_root, 'active repo before runner')
if 'MIRROR_REVISION_ROOT' in globals():
    _print_resnet_artifact_status(MIRROR_REVISION_ROOT, 'stable Drive mirror before runner')

resnet_should_run = (
    RUN_RESNET1D_CNN_BASELINE is True
    or (
        str(RUN_RESNET1D_CNN_BASELINE).strip().lower() == 'auto'
        and (bool(missing_resnet_baseline_artifacts) or bool(RESNET_FORCE_RERUN))
    )
)
print('Resolved RUN_RESNET1D_CNN_BASELINE:', RUN_RESNET1D_CNN_BASELINE, '| should_run=', resnet_should_run)
if resnet_should_run:
    run(resnet_command, log_path='reports/revision/logs/baseline_resnet1d_cnn.log')
    _print_resnet_artifact_status(revision_root, 'active repo after runner')
    if 'MIRROR_REVISION_ROOT' in globals():
        run(
            f'python -u scripts/revision/artifact_mirror.py publish --mirror-root "{MIRROR_REVISION_ROOT}"',
            log_path='reports/revision/logs/baseline_resnet1d_cnn_immediate_mirror_publish.log',
        )
        _print_resnet_artifact_status(MIRROR_REVISION_ROOT, 'stable Drive mirror after immediate publish')
    else:
        print('Immediate mirror publish skipped because MIRROR_REVISION_ROOT is not defined.')
else:
    print('ResNet1D/CNN baseline artifacts are reusable; skipping runner. Set RUN_RESNET1D_CNN_BASELINE=True to force execution.')


## Raw Mamba Baseline Runner

Optional fair architecture comparator. This trains the ECG-RAMBA raw-signal Mamba backbone from scratch with MiniRocket/PCA and HRV structurally removed. It is not an inference ablation of the full checkpoint. Use A100 High-RAM for the full run.

For Colab stability, run one fold at a time by setting `RAW_MAMBA_ONLY_FOLDS = '1'`, then `'2'`, ... `'5'`. Each subset run writes a fold cache and immediately publishes it to the Drive mirror. After all five fold caches exist, set `RAW_MAMBA_ONLY_FOLDS = ''` and rerun this cell once to aggregate canonical OOF predictions, summary, manifest, and paired evidence.

If `mamba_ssm` or `causal_conv1d` is missing, the runner cell reuses the canonical installer from notebook 02. Colab may intentionally restart after pinning torch; reconnect and rerun notebook 04 from the setup cell.


In [ ]:
RUN_RAW_MAMBA_BASELINE = 'auto'
RAW_MAMBA_EPOCHS = 20
RAW_MAMBA_BATCH_SIZE = 64
RAW_MAMBA_NUM_WORKERS = 2
RAW_MAMBA_N_BOOT = 1000
RAW_MAMBA_LIMIT_RECORDS = 0
RAW_MAMBA_ONLY_FOLDS = 'auto'  # auto runs only missing fold caches; empty string aggregates/reuses
RAW_MAMBA_FORCE_RERUN = 'auto'
RAW_MAMBA_SAVE_CHECKPOINTS = True
RAW_MAMBA_RETRAIN_FOR_MISSING_CHECKPOINTS = False  # Keep False to reuse completed baseline caches; True only for robustness checkpoint export.
RAW_MAMBA_FOLD_CACHE_RELPATHS = {
    fold: Path(f'predictions/folds/raw_mamba_fold{fold}_predictions.npz')
    for fold in range(1, 6)
}
RAW_MAMBA_BASELINE_RELPATHS = [
    Path('predictions/raw_mamba_oof_predictions.npz'),
    Path('predictions/raw_mamba_slice_predictions.npz'),
    Path('metrics/raw_mamba_baseline_summary.json'),
    Path('metrics/raw_mamba_fold_cache_status.json'),
    Path('tables/table_raw_mamba_class_metrics.csv'),
    Path('tables/table_raw_mamba_fold_summary.csv'),
    Path('tables/table_raw_mamba_fold_cache_status.csv'),
    Path('manifests/raw_mamba_baseline_manifest.json'),
    *RAW_MAMBA_FOLD_CACHE_RELPATHS.values(),
]
RAW_MAMBA_CHECKPOINT_RELPATHS = {
    fold: Path(f'experimental/raw_mamba_checkpoints/fold{fold}_raw_mamba_final_ema.pt')
    for fold in range(1, 6)
}
missing_raw_mamba_fold_cache_folds = [
    fold for fold, rel in RAW_MAMBA_FOLD_CACHE_RELPATHS.items()
    if not (revision_root / rel).exists() or (revision_root / rel).stat().st_size == 0
]
missing_raw_mamba_baseline_artifacts = [
    str(rel) for rel in RAW_MAMBA_BASELINE_RELPATHS
    if not (revision_root / rel).exists() or (revision_root / rel).stat().st_size == 0
]
missing_raw_mamba_checkpoint_folds = [
    fold for fold, rel in RAW_MAMBA_CHECKPOINT_RELPATHS.items()
    if not (revision_root / rel).exists() or (revision_root / rel).stat().st_size == 0
]
if str(RAW_MAMBA_ONLY_FOLDS).strip().lower() == 'auto':
    if missing_raw_mamba_fold_cache_folds:
        RAW_MAMBA_ONLY_FOLDS = ','.join(str(fold) for fold in missing_raw_mamba_fold_cache_folds)
    elif missing_raw_mamba_checkpoint_folds and RAW_MAMBA_RETRAIN_FOR_MISSING_CHECKPOINTS:
        RAW_MAMBA_ONLY_FOLDS = ','.join(str(fold) for fold in missing_raw_mamba_checkpoint_folds)
    else:
        RAW_MAMBA_ONLY_FOLDS = ''
print('Resolved RAW_MAMBA_ONLY_FOLDS:', RAW_MAMBA_ONLY_FOLDS or 'none; aggregate/reuse mode')
print('Missing Raw Mamba baseline artifacts:', missing_raw_mamba_baseline_artifacts or 'none')
print('Missing Raw Mamba checkpoint folds:', missing_raw_mamba_checkpoint_folds or 'none')
if missing_raw_mamba_checkpoint_folds and not RAW_MAMBA_RETRAIN_FOR_MISSING_CHECKPOINTS:
    print('Raw Mamba checkpoints are missing, but baseline outputs are reusable. Set RAW_MAMBA_RETRAIN_FOR_MISSING_CHECKPOINTS=True only if Notebook 05 learned-comparator stress needs checkpoints.')
if RAW_MAMBA_FORCE_RERUN == 'auto':
    RAW_MAMBA_FORCE_RERUN = bool(str(RAW_MAMBA_ONLY_FOLDS).strip())
    print('Resolved RAW_MAMBA_FORCE_RERUN:', RAW_MAMBA_FORCE_RERUN)

raw_mamba_command = (
    'python -u scripts/revision/16_raw_mamba_baseline.py '
    '--oof-predictions reports/revision/predictions/oof_final_ema_predictions.npz '
    '--freeze-manifest reports/revision/manifests/oof_final_ema_freeze_manifest.json '
    '--expected-checkpoint-kind final_ema '
    f'--epochs {RAW_MAMBA_EPOCHS} --batch-size {RAW_MAMBA_BATCH_SIZE} --num-workers {RAW_MAMBA_NUM_WORKERS} '
    '--threshold 0.5 --n-bins 15 '
    f'--n-boot {RAW_MAMBA_N_BOOT} '
    '--bce-pos-weight fold '
    '--device cuda --amp --allow-tf32 --reuse-predictions'
)
raw_mamba_aggregate_command = raw_mamba_command
if RAW_MAMBA_FORCE_RERUN:
    raw_mamba_command += ' --force-rerun'
if RAW_MAMBA_SAVE_CHECKPOINTS:
    raw_mamba_command += ' --save-checkpoints --checkpoint-dir reports/revision/experimental/raw_mamba_checkpoints'

if RAW_MAMBA_LIMIT_RECORDS:
    raw_mamba_command += f' --limit-records {RAW_MAMBA_LIMIT_RECORDS}'
    raw_mamba_aggregate_command += f' --limit-records {RAW_MAMBA_LIMIT_RECORDS}'
raw_mamba_fold_base_command = raw_mamba_command
raw_mamba_log_path = 'reports/revision/logs/baseline_raw_mamba.log'
if str(RAW_MAMBA_ONLY_FOLDS).strip():
    raw_mamba_command += f' --only-folds "{RAW_MAMBA_ONLY_FOLDS}"'
    fold_suffix = str(RAW_MAMBA_ONLY_FOLDS).replace(',', '_').replace(' ', '')
    raw_mamba_log_path = f'reports/revision/logs/baseline_raw_mamba_folds_{fold_suffix}.log'

raw_mamba_artifact_relpaths = [
    *RAW_MAMBA_BASELINE_RELPATHS,
    *RAW_MAMBA_CHECKPOINT_RELPATHS.values(),
]

def _print_raw_mamba_artifact_status(root, label):
    root = Path(root)
    print()
    print(f'Raw Mamba artifact status: {label} -> {root}')
    for rel in raw_mamba_artifact_relpaths:
        path = root / rel
        print(f'  {rel}: exists={path.exists()} size={path.stat().st_size if path.exists() else None}')

print('Raw Mamba baseline runner trains from scratch on raw ECG only with no MiniRocket, HRV, PCA, or full-model checkpoint weights.')
print('Completed folds are cached under reports/revision/predictions/folds/ and reused on rerun if present in the active repo.')
print('Checkpoint export for robustness requires RAW_MAMBA_RETRAIN_FOR_MISSING_CHECKPOINTS=True when checkpoints are absent; otherwise completed prediction caches are reused.')
print('Fold-cache mode:', RAW_MAMBA_ONLY_FOLDS or 'disabled; final aggregation/reuse mode')
print('After a successful run this cell immediately publishes reports/revision to the Drive mirror to survive Colab disconnects.')
print('Raw Mamba: implemented=', Path('scripts/revision/16_raw_mamba_baseline.py').exists(), 'planned_command=', raw_mamba_command)
_print_raw_mamba_artifact_status(revision_root, 'active repo before runner')
if 'MIRROR_REVISION_ROOT' in globals():
    _print_raw_mamba_artifact_status(MIRROR_REVISION_ROOT, 'stable Drive mirror before runner')



def ensure_mamba_runtime_for_raw_mamba():
    """Install/check the exact Mamba runtime before launching the raw Mamba comparator."""
    import importlib.util
    import json as _json

    missing = [module for module in ['mamba_ssm', 'causal_conv1d'] if importlib.util.find_spec(module) is None]
    if not missing:
        print('Mamba runtime available:', ', '.join(['mamba_ssm', 'causal_conv1d']))
        return

    if 'DRIVE_ROOT' not in globals():
        raise RuntimeError('DRIVE_ROOT is not defined. Run the notebook setup cell before Raw Mamba.')

    installer_notebook = Path('notebooks/02_predictions_and_external_eval.ipynb')
    if not installer_notebook.exists():
        raise FileNotFoundError(
            'Missing notebooks/02_predictions_and_external_eval.ipynb. Pull latest main before running Raw Mamba.'
        )

    nb02 = _json.loads(installer_notebook.read_text(encoding='utf-8'))
    installer_source = None
    for candidate in nb02.get('cells', []):
        candidate_source = ''.join(candidate.get('source', []))
        if (
            'Mamba wheel environment' in candidate_source
            and 'mamba_ssm' in candidate_source
            and 'causal_conv1d' in candidate_source
            and 'AUTO_PIN_TORCH_FOR_MAMBA' in candidate_source
        ):
            installer_source = candidate_source
            break

    if installer_source is None:
        raise RuntimeError('Could not locate the canonical Mamba installer cell in notebook 02.')

    print('Mamba runtime missing before Raw Mamba:', missing)
    print('Running canonical Mamba installer from Notebook 02. If torch is pinned, Colab will restart intentionally; rerun Notebook 04 from setup after reconnect.')
    exec(
        compile(installer_source, str(installer_notebook) + ':mamba_runtime_installer', 'exec'),
        globals(),
        globals(),
    )

    missing_after = [module for module in ['mamba_ssm', 'causal_conv1d'] if importlib.util.find_spec(module) is None]
    if missing_after:
        raise ImportError(
            'Mamba runtime is still missing after installer: '
            + ', '.join(missing_after)
            + '. If Colab restarted or torch was changed, reconnect and rerun Notebook 04 from the setup cell.'
        )

raw_mamba_should_run = (
    RUN_RAW_MAMBA_BASELINE is True
    or (
        str(RUN_RAW_MAMBA_BASELINE).strip().lower() == 'auto'
        and (bool(missing_raw_mamba_baseline_artifacts) or bool(RAW_MAMBA_FORCE_RERUN))
    )
)
print('Resolved RUN_RAW_MAMBA_BASELINE:', RUN_RAW_MAMBA_BASELINE, '| should_run=', raw_mamba_should_run)

def _publish_raw_mamba_mirror(label, log_path):
    _print_raw_mamba_artifact_status(revision_root, f'active repo {label}')
    if 'MIRROR_REVISION_ROOT' in globals():
        run(
            f'python -u scripts/revision/artifact_mirror.py publish --mirror-root "{MIRROR_REVISION_ROOT}"',
            log_path=log_path,
        )
        _print_raw_mamba_artifact_status(MIRROR_REVISION_ROOT, f'stable Drive mirror {label}')
    else:
        print('Mirror publish skipped because MIRROR_REVISION_ROOT is not defined.')

if raw_mamba_should_run:
    ensure_mamba_runtime_for_raw_mamba()
    selected_raw_mamba_folds = str(RAW_MAMBA_ONLY_FOLDS).strip()
    selected_raw_mamba_fold_tokens = [token.strip() for token in selected_raw_mamba_folds.split(',') if token.strip()]
    if selected_raw_mamba_fold_tokens:
        for fold_token in selected_raw_mamba_fold_tokens:
            if not fold_token.isdigit():
                raise ValueError(f'RAW_MAMBA_ONLY_FOLDS must contain fold numbers, got: {RAW_MAMBA_ONLY_FOLDS!r}')
            fold_command = raw_mamba_fold_base_command + f' --only-folds "{fold_token}"'
            fold_log = f'reports/revision/logs/baseline_raw_mamba_fold{fold_token}.log'
            print(f'Running Raw Mamba fold {fold_token} with immediate post-fold mirror publish.')
            run(fold_command, log_path=fold_log)
            _publish_raw_mamba_mirror(
                f'after fold {fold_token}',
                f'reports/revision/logs/baseline_raw_mamba_fold{fold_token}_mirror_publish.log',
            )
        raw_mamba_fold_cache_paths = [
            revision_root / 'predictions' / 'folds' / f'raw_mamba_fold{fold}_predictions.npz'
            for fold in range(1, 6)
        ]
        missing_raw_mamba_fold_caches = [str(path) for path in raw_mamba_fold_cache_paths if not path.exists()]
        if missing_raw_mamba_fold_caches:
            print('Raw Mamba fold-cache pass complete, but canonical aggregation is deferred because these fold caches are still missing:')
            for path in missing_raw_mamba_fold_caches:
                print(' -', path)
            print('Rerun this cell after the missing fold-cache run(s); auto mode will select only the remaining folds.')
        else:
            print('All Raw Mamba fold caches are present; running canonical aggregate/reuse pass now.')
            run(raw_mamba_aggregate_command, log_path='reports/revision/logs/baseline_raw_mamba_aggregate.log')
            _publish_raw_mamba_mirror('after aggregate pass', 'reports/revision/logs/baseline_raw_mamba_aggregate_mirror_publish.log')
    else:
        run(raw_mamba_aggregate_command, log_path='reports/revision/logs/baseline_raw_mamba_aggregate.log')
        _publish_raw_mamba_mirror('after aggregate/reuse pass', 'reports/revision/logs/baseline_raw_mamba_aggregate_mirror_publish.log')
else:
    print('Raw Mamba baseline artifacts are reusable; skipping runner. Set RUN_RAW_MAMBA_BASELINE=True to force execution.')


## Transformer ECG Baseline Runner

Optional reviewer-driven transformer comparator under the same frozen OOF/raw-ECG protocol. This is disabled by default because ResNet1D/CNN and Raw Mamba already provide strong learned architecture baselines. Run only if the response needs a transformer-specific comparator.


In [ ]:
RUN_TRANSFORMER_ECG_BASELINE = 'auto'
TRANSFORMER_EPOCHS = 20
TRANSFORMER_BATCH_SIZE = 256
TRANSFORMER_NUM_WORKERS = 2
TRANSFORMER_N_BOOT = 1000
TRANSFORMER_LR = 1e-3
TRANSFORMER_WEIGHT_DECAY = 1e-4
TRANSFORMER_DROPOUT = 0.20
TRANSFORMER_EMBED_DIM = 96
TRANSFORMER_FORCE_RERUN = False
TRANSFORMER_SAVE_CHECKPOINTS = True

transformer_command = (
    'python -u scripts/revision/24_transformer_ecg_baseline.py '
    '--oof-predictions reports/revision/predictions/oof_final_ema_predictions.npz '
    '--freeze-manifest reports/revision/manifests/oof_final_ema_freeze_manifest.json '
    '--expected-checkpoint-kind final_ema '
    f'--epochs {TRANSFORMER_EPOCHS} --batch-size {TRANSFORMER_BATCH_SIZE} --num-workers {TRANSFORMER_NUM_WORKERS} '
    f'--threshold 0.5 --n-bins 15 --n-boot {TRANSFORMER_N_BOOT} '
    f'--lr {TRANSFORMER_LR} --weight-decay {TRANSFORMER_WEIGHT_DECAY} --dropout {TRANSFORMER_DROPOUT} '
    f'--base-channels {TRANSFORMER_EMBED_DIM} --device cuda --amp --allow-tf32 --reuse-predictions'
)
if TRANSFORMER_FORCE_RERUN:
    transformer_command += ' --force-rerun'
if TRANSFORMER_SAVE_CHECKPOINTS:
    transformer_command += ' --save-checkpoints --checkpoint-dir reports/revision/experimental/transformer_ecg_checkpoints'

TRANSFORMER_BASELINE_RELPATHS = [
    'metrics/transformer_ecg_baseline_summary.json',
    'predictions/transformer_ecg_oof_predictions.npz',
    'predictions/transformer_ecg_slice_predictions.npz',
    'tables/table_transformer_ecg_class_metrics.csv',
    'tables/table_transformer_ecg_fold_summary.csv',
    'manifests/transformer_ecg_baseline_manifest.json',
    *[f'predictions/folds/transformer_ecg_fold{i}_predictions.npz' for i in range(1, 6)],
]
TRANSFORMER_CHECKPOINT_RELPATHS = [
    f'experimental/transformer_ecg_checkpoints/fold{i}_transformer_ecg_final.pt'
    for i in range(1, 6)
]
missing_transformer_baseline_artifacts = [
    rel for rel in TRANSFORMER_BASELINE_RELPATHS
    if not (revision_root / rel).exists() or (revision_root / rel).stat().st_size == 0
]
transformer_should_run = (
    RUN_TRANSFORMER_ECG_BASELINE is True
    or (str(RUN_TRANSFORMER_ECG_BASELINE).strip().lower() == 'auto' and bool(missing_transformer_baseline_artifacts))
)
print('Transformer ECG baseline runner is fold-safe and raw-ECG-only. It reuses the ResNet baseline data pipeline and swaps only the model class.')
print('Checkpoint export is enabled so interrupted runs can be audited/reused if needed.')
print('Transformer ECG command:', transformer_command)
print('Transformer runner implemented=', Path('scripts/revision/24_transformer_ecg_baseline.py').exists())
print('Missing Transformer baseline artifacts:', missing_transformer_baseline_artifacts or 'none')
print('Resolved RUN_TRANSFORMER_ECG_BASELINE:', RUN_TRANSFORMER_ECG_BASELINE, '| should_run=', transformer_should_run)
if transformer_should_run:
    run(transformer_command, log_path='reports/revision/logs/baseline_transformer_ecg.log')
    if 'MIRROR_REVISION_ROOT' in globals():
        run(
            f'python -u scripts/revision/artifact_mirror.py publish --mirror-root "{MIRROR_REVISION_ROOT}"',
            log_path='reports/revision/logs/baseline_transformer_ecg_mirror_publish.log',
        )
else:
    print('Transformer ECG baseline artifacts are reusable; skipping runner. Set RUN_TRANSFORMER_ECG_BASELINE=True to force execution.')

for rel in [*TRANSFORMER_BASELINE_RELPATHS, *TRANSFORMER_CHECKPOINT_RELPATHS]:
    path = revision_root / rel
    print(f'{path}: exists={path.exists()} size={path.stat().st_size if path.exists() else None}')


## Hybrid MiniRocket Morphology MLP Runner

Optional reviewer-driven morphology sensitivity baseline. It reuses the fold-safe MiniRocket feature cache but replaces the linear head with a small learnable MLP. Treat this only as a determinism-vs-learnable-head sensitivity check, not as proof of mechanism or disentanglement.


In [ ]:
RUN_HYBRID_MORPHOLOGY_BASELINE = 'auto'
HYBRID_MORPHOLOGY_EPOCHS = 20
HYBRID_MORPHOLOGY_BATCH_SIZE = 4096
HYBRID_MORPHOLOGY_STATS_BATCH_SIZE = 1024
HYBRID_MORPHOLOGY_N_BOOT = 1000
HYBRID_MORPHOLOGY_LR = 1e-3
HYBRID_MORPHOLOGY_WEIGHT_DECAY = 1e-4
HYBRID_MORPHOLOGY_HIDDEN_DIM = 512
HYBRID_MORPHOLOGY_DROPOUT = 0.20
HYBRID_MORPHOLOGY_FORCE_RERUN = False
HYBRID_MORPHOLOGY_LIMIT_RECORDS = 0

hybrid_command = (
    'python -u scripts/revision/26_hybrid_morphology_baseline.py '
    '--oof-predictions reports/revision/predictions/oof_final_ema_predictions.npz '
    '--freeze-manifest reports/revision/manifests/oof_final_ema_freeze_manifest.json '
    '--expected-checkpoint-kind final_ema '
    f'--epochs {HYBRID_MORPHOLOGY_EPOCHS} --batch-size {HYBRID_MORPHOLOGY_BATCH_SIZE} '
    f'--stats-batch-size {HYBRID_MORPHOLOGY_STATS_BATCH_SIZE} '
    '--threshold 0.5 --n-bins 15 '
    f'--n-boot {HYBRID_MORPHOLOGY_N_BOOT} '
    f'--lr {HYBRID_MORPHOLOGY_LR} --weight-decay {HYBRID_MORPHOLOGY_WEIGHT_DECAY} '
    f'--hidden-dim {HYBRID_MORPHOLOGY_HIDDEN_DIM} --dropout {HYBRID_MORPHOLOGY_DROPOUT} '
    '--device cuda --standardize train_fold --allow-tf32'
)
if not HYBRID_MORPHOLOGY_FORCE_RERUN:
    hybrid_command += ' --reuse-predictions'
if HYBRID_MORPHOLOGY_LIMIT_RECORDS:
    hybrid_command += f' --limit-records {HYBRID_MORPHOLOGY_LIMIT_RECORDS}'

HYBRID_MORPHOLOGY_BASELINE_RELPATHS = [
    'metrics/hybrid_morphology_baseline_summary.json',
    'predictions/hybrid_morphology_oof_predictions.npz',
    'tables/table_hybrid_morphology_class_metrics.csv',
    'tables/table_hybrid_morphology_fold_summary.csv',
    'manifests/hybrid_morphology_baseline_manifest.json',
]
missing_hybrid_morphology_artifacts = [
    rel for rel in HYBRID_MORPHOLOGY_BASELINE_RELPATHS
    if not (revision_root / rel).exists() or (revision_root / rel).stat().st_size == 0
]
hybrid_morphology_should_run = (
    RUN_HYBRID_MORPHOLOGY_BASELINE is True
    or (str(RUN_HYBRID_MORPHOLOGY_BASELINE).strip().lower() == 'auto' and bool(missing_hybrid_morphology_artifacts))
)
print('Hybrid MiniRocket morphology MLP command:', hybrid_command)
print('Hybrid runner implemented=', Path('scripts/revision/26_hybrid_morphology_baseline.py').exists())
print('Missing Hybrid morphology artifacts:', missing_hybrid_morphology_artifacts or 'none')
print('Resolved RUN_HYBRID_MORPHOLOGY_BASELINE:', RUN_HYBRID_MORPHOLOGY_BASELINE, '| should_run=', hybrid_morphology_should_run)
if hybrid_morphology_should_run:
    run(hybrid_command, log_path='reports/revision/logs/baseline_hybrid_morphology.log')
    if 'MIRROR_REVISION_ROOT' in globals():
        run(
            f'python -u scripts/revision/artifact_mirror.py publish --mirror-root "{MIRROR_REVISION_ROOT}"',
            log_path='reports/revision/logs/baseline_hybrid_morphology_mirror_publish.log',
        )
else:
    print('Hybrid morphology baseline artifacts are reusable; skipping runner. Set RUN_HYBRID_MORPHOLOGY_BASELINE=True to force execution.')

for rel in HYBRID_MORPHOLOGY_BASELINE_RELPATHS:
    path = revision_root / rel
    print(f'{path}: exists={path.exists()} size={path.stat().st_size if path.exists() else None}')


## Fair Baseline Completion Matrix

This table is the reviewer-facing baseline ledger. A completed row must have the same split, preprocessing, threshold, and record-level protocol. Rows marked blocked must not be used to claim superiority over fair baselines.


In [ ]:
if 'restore_available_revision_artifacts' in globals():
    optional_artifacts_to_restore = globals().get('OPTIONAL_BASELINE_ARTIFACTS_FOR_04', NOTEBOOK05_AVAILABLE_ARTIFACTS_FOR_04)
    if globals().get('RUN_MINIROCKET_ONLY_BASELINE', False):
        optional_artifacts_to_restore = [
            rel for rel in optional_artifacts_to_restore
            if rel not in globals().get('MINIROCKET_AVAILABLE_ARTIFACTS_FOR_04', [])
        ]
        print('Skipping MiniRocket artifact restore from mirror because this notebook just generated them in the active repo.')
    if globals().get('RUN_RESNET1D_CNN_BASELINE', False):
        optional_artifacts_to_restore = [
            rel for rel in optional_artifacts_to_restore
            if rel not in globals().get('RESNET1D_CNN_AVAILABLE_ARTIFACTS_FOR_04', [])
        ]
        print('Skipping ResNet1D/CNN artifact restore from mirror because this notebook just generated them in the active repo.')
    if globals().get('RUN_RAW_MAMBA_BASELINE', False):
        optional_artifacts_to_restore = [
            rel for rel in optional_artifacts_to_restore
            if rel not in globals().get('RAW_MAMBA_AVAILABLE_ARTIFACTS_FOR_04', [])
        ]
        print('Skipping Raw Mamba artifact restore from mirror because this notebook just generated them in the active repo.')
    if globals().get('RUN_TRANSFORMER_ECG_BASELINE', False):
        optional_artifacts_to_restore = [
            rel for rel in optional_artifacts_to_restore
            if rel not in globals().get('TRANSFORMER_ECG_AVAILABLE_ARTIFACTS_FOR_04', [])
        ]
        print('Skipping Transformer ECG artifact restore from mirror because this notebook just generated them in the active repo.')
    if globals().get('RUN_HYBRID_MORPHOLOGY_BASELINE', False):
        optional_artifacts_to_restore = [
            rel for rel in optional_artifacts_to_restore
            if rel not in globals().get('HYBRID_MORPHOLOGY_AVAILABLE_ARTIFACTS_FOR_04', [])
        ]
        print('Skipping Hybrid morphology artifact restore from mirror because this notebook just generated them in the active repo.')
    restore_available_revision_artifacts(optional_artifacts_to_restore)

metric_names = [
    'roc_auc_macro', 'pr_auc_macro', 'f1_macro', 'f1_micro', 'precision_macro',
    'recall_macro', 'sensitivity_macro', 'specificity_macro', 'ppv_macro', 'npv_macro',
    'brier_macro', 'ece_macro', 'ece_max', 'mce_macro', 'mce_max'
]
full_metrics = {name: calibration.get('metrics', {}).get(name) for name in metric_names}
full_metrics.update({name: calibration.get('calibration', {}).get(name) for name in metric_names if name not in full_metrics or full_metrics[name] is None})
bootstrap_ci = calibration.get('bootstrap_ci', {})
ci_metric_map = {
    'roc_auc_macro': 'macro_roc_auc',
    'pr_auc_macro': 'macro_pr_auc',
    'f1_macro': 'f1_macro',
    'brier_macro': 'brier_macro',
    'ece_macro': 'ece_macro',
}

import shutil

def _read_summary_json(path):
    path = Path(path)
    if not path.exists():
        return None
    try:
        return json.loads(path.read_text(encoding='utf-8'))
    except Exception as exc:
        return {'_read_error': repr(exc)}

def _candidate_paths(relative_path):
    candidates = [('active_repo', revision_root / relative_path)]
    if 'MIRROR_REVISION_ROOT' in globals():
        candidates.append(('stable_drive_mirror', MIRROR_REVISION_ROOT / relative_path))
    if 'DRIVE_ROOT' in globals():
        candidates.append(('drive_repo', DRIVE_ROOT / 'ECG-RAMBA' / 'reports' / 'revision' / relative_path))
    return candidates

def _feature_summary_report(label, path, *, expected_feature_contract, expected_cache_kind_key, expected_protocol=None, expected_feature_preprocessing=None):
    path = Path(path)
    summary = _read_summary_json(path)
    load_info = summary.get('load_info', {}) if isinstance(summary, dict) else {}
    freeze_contract = load_info.get('freeze_contract') or {}
    expected_records = int(freeze.get('validated_records', -1))
    expected_classes = int(freeze.get('n_classes', -1))
    expected_fingerprint = freeze.get('dataset_record_order_fingerprint')
    expected_fold_counts = {str(k): int(v) for k, v in (freeze.get('fold_counts') or {}).items()}
    observed_fold_counts = {str(k): int(v) for k, v in (load_info.get('fold_counts') or freeze_contract.get('fold_counts') or {}).items()}
    observed_fingerprint = (
        load_info.get('dataset_record_order_fingerprint')
        or freeze_contract.get('dataset_record_order_fingerprint')
    )
    exact_oof_sha_match = load_info.get('oof_predictions_sha256') == pred_sha
    protocol_compatible_oof = (
        int(load_info.get('oof_records_used', -1)) == expected_records
        and int(load_info.get('oof_records_total', expected_records)) == expected_records
        and (not expected_fingerprint or observed_fingerprint == expected_fingerprint)
        and (not expected_fold_counts or observed_fold_counts == expected_fold_counts)
        and freeze_contract.get('checkpoint_kind') == 'final_ema'
    )
    # Baseline artifacts are independent comparator predictions. When Notebook 02
    # rewrites the Full OOF NPZ container, the file SHA can change even if record
    # order, labels, folds, and checkpoint contract are unchanged. Reuse such
    # comparator artifacts only when the stable frozen protocol contract matches;
    # paired comparison cells below still rerun against the current Full OOF SHA.
    valid = (
        isinstance(summary, dict)
        and '_read_error' not in summary
        and summary.get('feature_contract') == expected_feature_contract
        and (expected_protocol is None or summary.get('protocol') == expected_protocol)
        and (expected_feature_preprocessing is None or summary.get('feature_preprocessing') == expected_feature_preprocessing)
        and int(summary.get('n_records', -1)) == expected_records
        and int(summary.get('n_classes', -1)) == expected_classes
        and summary.get('manuscript_ready', True) is True
        and (exact_oof_sha_match or protocol_compatible_oof)
        and load_info.get(expected_cache_kind_key) == 'record_fingerprinted'
        and freeze_contract.get('checkpoint_kind') == 'final_ema'
    )
    return {
        'label': label,
        'path': path,
        'exists': path.exists(),
        'size': path.stat().st_size if path.exists() else None,
        'valid': valid,
        'summary': summary if isinstance(summary, dict) and '_read_error' not in summary else None,
        'read_error': summary.get('_read_error') if isinstance(summary, dict) else None,
        'feature_contract': summary.get('feature_contract') if isinstance(summary, dict) else None,
        'protocol': summary.get('protocol') if isinstance(summary, dict) else None,
        'feature_preprocessing': summary.get('feature_preprocessing') if isinstance(summary, dict) else None,
        'n_records': summary.get('n_records') if isinstance(summary, dict) else None,
        'n_classes': summary.get('n_classes') if isinstance(summary, dict) else None,
        'manuscript_ready': summary.get('manuscript_ready', True) if isinstance(summary, dict) else None,
        'oof_records_used': load_info.get('oof_records_used'),
        'oof_predictions_sha256': load_info.get('oof_predictions_sha256'),
        'cache_kind': load_info.get(expected_cache_kind_key),
        'freeze_checkpoint_kind': freeze_contract.get('checkpoint_kind'),
    }

def _select_feature_artifact(name, *, summary_relative_path, artifact_rel_paths, expected_feature_contract, expected_cache_kind_key, expected_protocol=None, expected_feature_preprocessing=None):
    reports = [
        _feature_summary_report(
            label,
            path,
            expected_feature_contract=expected_feature_contract,
            expected_cache_kind_key=expected_cache_kind_key,
            expected_protocol=expected_protocol,
            expected_feature_preprocessing=expected_feature_preprocessing,
        )
        for label, path in _candidate_paths(summary_relative_path)
    ]

    for report in reports:
        report['missing_artifacts'] = []
        if report['valid']:
            source_revision_root = report['path'].parents[1]
            missing = [str(source_revision_root / rel) for rel in artifact_rel_paths if not (source_revision_root / rel).exists()]
            if missing:
                report['missing_artifacts'] = missing
                report['valid'] = False

    print(f'{name} artifact contract check:')
    print('  expected final_ema prediction sha256:', pred_sha)
    for report in reports:
        print(
            f"  {report['label']}: exists={report['exists']} valid={report['valid']} "
            f"size={report['size']} feature={report['feature_contract']} "
            f"protocol={report['protocol']} preprocessing={report['feature_preprocessing']} "
            f"n_records={report['n_records']} used={report['oof_records_used']} "
            f"manuscript_ready={report['manuscript_ready']} "
            f"oof_sha={report['oof_predictions_sha256']} current_oof_sha={pred_sha} cache_kind={report['cache_kind']} "
            f"freeze_kind={report['freeze_checkpoint_kind']} missing_artifacts={len(report['missing_artifacts'])} "
            f"path={report['path']}"
        )
        if report['read_error']:
            print(f"    read_error={report['read_error']}")
        if report['missing_artifacts']:
            print('    missing artifact examples:', report['missing_artifacts'][:5])

    valid_reports = [report for report in reports if report['valid']]
    active_path = revision_root / summary_relative_path
    if valid_reports:
        chosen = valid_reports[0]
        if chosen['path'] != active_path:
            source_revision_root = chosen['path'].parents[1]
            copied = []
            for rel in artifact_rel_paths:
                source = source_revision_root / rel
                destination = revision_root / rel
                if source.exists():
                    destination.parent.mkdir(parents=True, exist_ok=True)
                    shutil.copy2(source, destination)
                    copied.append(rel)
            print(f'Restored canonical {name} artifacts into active repo from:', chosen['label'])
            print('  source revision root:', source_revision_root)
            print('  copied artifacts:', len(copied))
        return chosen['summary'], True, ''

    active_summary = reports[0]['summary']
    if active_summary is not None:
        candidate_notes = '; '.join(
            f"{report['label']}: exists={report['exists']}, valid={report['valid']}, "
            f"feature={report['feature_contract']}, protocol={report['protocol']}, "
            f"preprocessing={report['feature_preprocessing']}, n_records={report['n_records']}, "
            f"used={report['oof_records_used']}, manuscript_ready={report['manuscript_ready']}, "
            f"oof_sha={report['oof_predictions_sha256']}, cache_kind={report['cache_kind']}, "
            f"freeze_kind={report['freeze_checkpoint_kind']}, missing_artifacts={len(report.get('missing_artifacts', []))}"
            for report in reports
        )
        stale_reason = (
            f'Existing {name} summary is stale or non-canonical. '
            f'Expected final_ema oof_predictions_sha256={pred_sha}, '
            f'feature_contract={expected_feature_contract}, protocol={expected_protocol or "<any>"}, '
            f'feature_preprocessing={expected_feature_preprocessing or "<any>"}, '
            f'n_records={int(freeze.get("validated_records", -1))}, '
            f'n_classes={int(freeze.get("n_classes", -1))}, manuscript_ready=true, record-fingerprinted cache, '
            f'and all expected companion artifacts present. Candidate status: {candidate_notes}.'
        )
        return active_summary, False, stale_reason
    return None, False, ''


hrv_only_summary_path = revision_root / 'metrics' / 'hrv_only_baseline_summary.json'
hrv_only_summary, hrv_only_artifact_valid, hrv_only_stale_reason = _select_feature_artifact(
    'HRV-only',
    summary_relative_path=Path('metrics/hrv_only_baseline_summary.json'),
    artifact_rel_paths=globals().get('HRV_ONLY_AVAILABLE_ARTIFACTS_FOR_04', NOTEBOOK05_AVAILABLE_ARTIFACTS_FOR_04),
    expected_feature_contract='hrv36',
    expected_cache_kind_key='chapman_hrv_cache_kind',
)

minirocket_summary_path = revision_root / 'metrics' / 'minirocket_only_baseline_summary.json'
minirocket_summary, minirocket_artifact_valid, minirocket_stale_reason = _select_feature_artifact(
    'MiniRocket-only',
    summary_relative_path=Path('metrics/minirocket_only_baseline_summary.json'),
    artifact_rel_paths=globals().get('MINIROCKET_AVAILABLE_ARTIFACTS_FOR_04', []),
    expected_feature_contract='minirocket_raw',
    expected_cache_kind_key='minirocket_cache_kind',
    expected_protocol='minirocket_raw_standardized_torch_linear_same_folds_threshold_0.5',
    expected_feature_preprocessing='fold_train_standardization',
)

resnet_summary_path = revision_root / 'metrics' / 'resnet1d_cnn_baseline_summary.json'
resnet_summary, resnet_artifact_valid, resnet_stale_reason = _select_feature_artifact(
    'ResNet1D/CNN',
    summary_relative_path=Path('metrics/resnet1d_cnn_baseline_summary.json'),
    artifact_rel_paths=globals().get('RESNET1D_CNN_AVAILABLE_ARTIFACTS_FOR_04', []),
    expected_feature_contract='raw_ecg_12lead',
    expected_cache_kind_key='raw_cache_kind',
    expected_protocol='resnet1d_cnn_raw_same_folds_power_mean_v2_q3_threshold_0.5',
)

raw_mamba_summary_path = revision_root / 'metrics' / 'raw_mamba_baseline_summary.json'
raw_mamba_summary, raw_mamba_artifact_valid, raw_mamba_stale_reason = _select_feature_artifact(
    'Raw Mamba',
    summary_relative_path=Path('metrics/raw_mamba_baseline_summary.json'),
    artifact_rel_paths=globals().get('RAW_MAMBA_AVAILABLE_ARTIFACTS_FOR_04', []),
    expected_feature_contract='raw_ecg_12lead_mamba_only',
    expected_cache_kind_key='raw_cache_kind',
    expected_protocol='raw_mamba_retrained_weighted_bce_same_folds_power_mean_v2_q3_threshold_0.5',
)

transformer_summary_path = revision_root / 'metrics' / 'transformer_ecg_baseline_summary.json'
transformer_summary, transformer_artifact_valid, transformer_stale_reason = _select_feature_artifact(
    'Transformer ECG',
    summary_relative_path=Path('metrics/transformer_ecg_baseline_summary.json'),
    artifact_rel_paths=globals().get('TRANSFORMER_ECG_AVAILABLE_ARTIFACTS_FOR_04', []),
    expected_feature_contract='raw_ecg_12lead',
    expected_cache_kind_key='raw_cache_kind',
    expected_protocol='transformer_ecg_raw_same_folds_power_mean_v2_q3_threshold_0.5',
)

fair_baseline_completion_flags = {
    'Raw Mamba': raw_mamba_artifact_valid,
    'ResNet1D/CNN': resnet_artifact_valid,
    'MiniRocket-only': minirocket_artifact_valid,
    'HRV-only': hrv_only_artifact_valid,
}
missing_required_fair_baselines = [name for name, complete in fair_baseline_completion_flags.items() if not complete]
missing_required_fair_text = ', '.join(missing_required_fair_baselines) if missing_required_fair_baselines else 'none'
missing_model_specific = ([] if raw_mamba_artifact_valid else ['Raw Mamba']) + ([] if resnet_artifact_valid else ['ResNet1D/CNN'])
missing_model_specific_text = ', '.join(missing_model_specific) if missing_model_specific else 'none'
fair_baseline_claim_reason = (
    'All required fair baseline rows are complete; broad superiority is still not automatic and must follow paired metric-specific deltas.'
    if not missing_required_fair_baselines
    else 'Fair-baseline superiority remains blocked until these rows are complete: ' + missing_required_fair_text + '.'
)

def baseline_row(name, family, status, evidence_path='', blocker='', notes='', metrics_payload=None, ci_payload=None, claim_reason=None):
    is_full = name == 'Full ECG-RAMBA frozen OOF'
    metrics_source = full_metrics if is_full else (metrics_payload or {})
    ci_source = bootstrap_ci if is_full else (ci_payload or {})
    row = {
        'baseline_name': name,
        'model_family': family,
        'dataset': 'chapman_oof',
        'split_protocol': 'same frozen five-fold OOF required',
        'threshold': THRESHOLD,
        'aggregation_protocol': 'record-level fixed threshold; Q=3 for full model where applicable',
        'status': status,
        'evidence_path': evidence_path,
        'blocker': blocker,
        'notes': notes,
        'claim_eligible_for_fair_baseline_superiority': False,
        'claim_eligibility_reason': claim_reason or ('Fair comparator rows are incomplete.' if is_full else 'Runner/result missing under the same protocol.'),
    }
    for metric in metric_names:
        row[metric] = metrics_source.get(metric, math.nan)
    for metric, ci_key in ci_metric_map.items():
        ci = ci_source.get(ci_key, {})
        row[f'{metric}_ci_low'] = ci.get('lo', math.nan)
        row[f'{metric}_ci_high'] = ci.get('hi', math.nan)
        row[f'{metric}_ci_mean'] = ci.get('mean', math.nan)
        row[f'{metric}_n_boot_valid'] = ci.get('n_boot_valid', math.nan)
    return row

def _completed_feature_row(name, summary, evidence_path, status, notes, claim_reason, family='feature_baseline'):
    return baseline_row(
        name,
        family,
        status,
        evidence_path=str(evidence_path),
        notes=notes,
        metrics_payload={**summary.get('metrics', {}), **summary.get('calibration', {})},
        ci_payload=summary.get('bootstrap_ci', {}),
        claim_reason=claim_reason,
    )

if hrv_only_artifact_valid:
    hrv_only_row = _completed_feature_row(
        'HRV-only',
        hrv_only_summary,
        hrv_only_summary_path,
        'complete_feature_baseline_from_notebook05',
        'Produced by scripts/revision/09_hrv_domain_analysis.py under the same frozen Chapman folds; report as HRV-only feature baseline, not full-model evidence.',
        fair_baseline_claim_reason,
    )
elif hrv_only_summary is not None:
    hrv_only_row = baseline_row(
        'HRV-only', 'feature_baseline', 'stale_artifact_rejected',
        evidence_path=str(hrv_only_summary_path),
        blocker=hrv_only_stale_reason,
        notes='The existing HRV-only artifact is intentionally not used because it failed the canonical artifact contract.',
        claim_reason='Rerun Notebook 05 against oof_final_ema_predictions.npz, then rerun Notebook 04.',
    )
else:
    hrv_only_row = baseline_row(
        'HRV-only', 'feature_baseline', 'pending_notebook05_hrv_runner',
        blocker='Run Notebook 05 / scripts/revision/09_hrv_domain_analysis.py to produce hrv_only_baseline_summary.json.',
        notes='Required before claiming fair baseline superiority or HRV-only comparison.',
    )

if minirocket_artifact_valid:
    minirocket_row = _completed_feature_row(
        'MiniRocket-only',
        minirocket_summary,
        minirocket_summary_path,
        'complete_feature_baseline_from_script10',
        'Produced by scripts/revision/10_minirocket_only_baseline.py using record-fingerprinted RAW MiniRocket features, fold-train standardization, fixed-epoch linear logistic heads, and the frozen final_ema OOF folds.',
        fair_baseline_claim_reason,
    )
elif minirocket_summary is not None:
    minirocket_row = baseline_row(
        'MiniRocket-only', 'feature_baseline', 'stale_artifact_rejected',
        evidence_path=str(minirocket_summary_path),
        blocker=minirocket_stale_reason,
        notes='The existing MiniRocket-only artifact is intentionally not used because it failed the canonical artifact contract.',
        claim_reason='Rerun scripts/revision/10_minirocket_only_baseline.py under Notebook 04, then rerun the baseline matrix cell.',
    )
else:
    minirocket_row = baseline_row(
        'MiniRocket-only', 'feature_baseline', 'pending_minirocket_runner',
        blocker='Run scripts/revision/10_minirocket_only_baseline.py from the Heavy Baseline Guard cell to produce minirocket_only_baseline_summary.json.',
        notes='Required before claiming fair baseline superiority or deterministic morphology-feature comparison.',
    )

if raw_mamba_artifact_valid:
    raw_mamba_row = _completed_feature_row(
        'Raw Mamba',
        raw_mamba_summary,
        raw_mamba_summary_path,
        'complete_architecture_baseline_from_script16',
        'Produced by scripts/revision/16_raw_mamba_baseline.py using raw 12-lead ECG slices, structural removal of MiniRocket/PCA and HRV streams, EMA final weights, Power Mean Q=3 aggregation, and frozen final_ema OOF folds.',
        fair_baseline_claim_reason,
        family='architecture_baseline',
    )
elif raw_mamba_summary is not None:
    raw_mamba_row = baseline_row(
        'Raw Mamba', 'architecture_baseline', 'stale_artifact_rejected',
        evidence_path=str(raw_mamba_summary_path),
        blocker=raw_mamba_stale_reason,
        notes='The existing Raw Mamba artifact is intentionally not used because it failed the canonical artifact contract.',
        claim_reason='Rerun scripts/revision/16_raw_mamba_baseline.py under Notebook 04, then rerun the baseline matrix cell.',
    )
elif Path('scripts/revision/16_raw_mamba_baseline.py').exists():
    raw_mamba_row = baseline_row(
        'Raw Mamba', 'architecture_baseline', 'implemented_pending_execution',
        blocker='Runner exists but no canonical raw_mamba_baseline_summary.json artifact has passed the frozen OOF contract yet.',
        notes='Set RUN_RAW_MAMBA_BASELINE=True in the Raw Mamba runner cell, then rerun this matrix cell.',
        claim_reason='Raw Mamba cannot close the fair-baseline gap until its artifact is generated and passes this contract.',
    )
else:
    raw_mamba_row = baseline_row(
        'Raw Mamba', 'architecture_baseline', 'blocked_runner_tbd',
        blocker='No implemented runner in scripts/revision; experiment_registry primary_script is TBD.',
        notes='Required before claiming fair baseline superiority over all fair comparators.',
    )

if resnet_artifact_valid:
    resnet_row = _completed_feature_row(
        'ResNet1D/CNN',
        resnet_summary,
        resnet_summary_path,
        'complete_architecture_baseline_from_script14',
        'Produced by scripts/revision/14_resnet1d_cnn_baseline.py using raw 12-lead ECG slices, fixed final epochs, Power Mean Q=3 aggregation, and frozen final_ema OOF folds. It does not use MiniRocket, HRV, PCA, or ECG-RAMBA checkpoints.',
        fair_baseline_claim_reason,
        family='architecture_baseline',
    )
elif resnet_summary is not None:
    resnet_row = baseline_row(
        'ResNet1D/CNN', 'architecture_baseline', 'stale_artifact_rejected',
        evidence_path=str(resnet_summary_path),
        blocker=resnet_stale_reason,
        notes='The existing ResNet1D/CNN artifact is intentionally not used because it failed the canonical artifact contract.',
        claim_reason='Rerun scripts/revision/14_resnet1d_cnn_baseline.py under Notebook 04, then rerun the baseline matrix cell.',
    )
elif Path('scripts/revision/14_resnet1d_cnn_baseline.py').exists():
    resnet_row = baseline_row(
        'ResNet1D/CNN', 'architecture_baseline', 'implemented_pending_execution',
        blocker='Runner exists but no canonical resnet1d_cnn_baseline_summary.json artifact has passed the frozen OOF contract yet.',
        notes='Set RUN_RESNET1D_CNN_BASELINE=True in the ResNet1D/CNN runner cell, then rerun this matrix cell.',
        claim_reason='ResNet1D/CNN cannot support fair-baseline claims until its artifact is generated and passes this contract.',
    )
else:
    resnet_row = baseline_row(
        'ResNet1D/CNN', 'architecture_baseline', 'blocked_runner_tbd',
        blocker='No implemented runner in scripts/revision; experiment_registry primary_script is TBD.',
        notes='Required before claiming fair baseline superiority.',
    )

if transformer_artifact_valid:
    transformer_row = _completed_feature_row(
        'Transformer ECG',
        transformer_summary,
        transformer_summary_path,
        'complete_optional_architecture_baseline_from_script24',
        'Produced by scripts/revision/24_transformer_ecg_baseline.py using raw 12-lead ECG slices, fixed final epochs, Power Mean Q=3 aggregation, and frozen final_ema OOF folds. This row is optional reviewer evidence and is not required for broad fair-baseline claims.',
        'Use only after paired bootstrap comparison; do not use this optional row to claim global superiority.',
        family='architecture_baseline_optional',
    )
elif transformer_summary is not None:
    transformer_row = baseline_row(
        'Transformer ECG', 'architecture_baseline_optional', 'stale_artifact_rejected',
        evidence_path=str(transformer_summary_path),
        blocker=transformer_stale_reason,
        notes='The existing Transformer ECG artifact is intentionally not used because it failed the canonical artifact contract.',
        claim_reason='Rerun scripts/revision/24_transformer_ecg_baseline.py under Notebook 04, then rerun this matrix cell.',
    )
elif Path('scripts/revision/24_transformer_ecg_baseline.py').exists():
    transformer_row = baseline_row(
        'Transformer ECG', 'architecture_baseline_optional', 'optional_runner_available_pending_execution',
        blocker='Optional runner exists but no canonical transformer_ecg_baseline_summary.json artifact has passed the frozen OOF contract yet.',
        notes='Set RUN_TRANSFORMER_ECG_BASELINE=True only if reviewer pressure requires a transformer-specific comparator.',
        claim_reason='Transformer ECG is optional evidence; paired comparison is required before using any significant delta wording.',
    )
else:
    transformer_row = baseline_row(
        'Transformer ECG', 'architecture_baseline_optional', 'optional_runner_missing',
        blocker='No Transformer ECG runner is available.',
        notes='Optional only; ResNet1D/CNN and Raw Mamba remain the required learned architecture comparators.',
    )

baseline_rows = [
    baseline_row(
        'Full ECG-RAMBA frozen OOF',
        'full_model',
        'complete_frozen_oof',
        evidence_path=str(calibration_path),
        notes='Validated against oof_final_ema_freeze_manifest.json and calibration_ci_oof_final_ema_predictions.json.',
        claim_reason=fair_baseline_claim_reason,
    ),
    raw_mamba_row,
    resnet_row,
    minirocket_row,
    hrv_only_row,
    transformer_row,
]

baseline_df = pd.DataFrame(baseline_rows)
baseline_csv = revision_root / 'metrics' / 'baseline_summary.csv'
baseline_table = revision_root / 'tables' / 'table_baselines.csv'
baseline_df.to_csv(baseline_csv, index=False)
baseline_df.to_csv(baseline_table, index=False)
print('Wrote:', baseline_csv)
print('Wrote:', baseline_table)
display(baseline_df)


## Paired Full vs MiniRocket Evidence

This cell turns the Full ECG-RAMBA vs MiniRocket-only baseline result into reviewer-ready paired evidence. It uses record-level paired bootstrap deltas, not independent CIs, so every resample compares both models on exactly the same records.


In [ ]:
RUN_PAIRED_FULL_VS_MINIROCKET = True
FORCE_RERUN_PAIRED_COMPARISON = False

paired_json = revision_root / 'metrics' / 'paired_full_vs_minirocket_comparison.json'
paired_table = revision_root / 'tables' / 'table_paired_full_vs_minirocket.csv'
paired_samples = revision_root / 'metrics' / 'paired_full_vs_minirocket_bootstrap_samples.csv'
paired_manifest = revision_root / 'manifests' / 'paired_full_vs_minirocket_manifest.json'

paired_inputs = {
    'full_predictions': revision_root / 'predictions' / 'oof_final_ema_predictions.npz',
    'minirocket_predictions': revision_root / 'predictions' / 'minirocket_only_oof_predictions.npz',
    'freeze_manifest': revision_root / 'manifests' / 'oof_final_ema_freeze_manifest.json',
    'minirocket_summary': revision_root / 'metrics' / 'minirocket_only_baseline_summary.json',
    'minirocket_manifest': revision_root / 'manifests' / 'minirocket_only_baseline_manifest.json',
}
for label, path in paired_inputs.items():
    print(f'{label:24}: exists={path.exists()} size={path.stat().st_size if path.exists() else None} path={path}')
missing_paired_inputs = [str(path) for path in paired_inputs.values() if not path.exists()]

from scripts.revision.common import sha256_file

def paired_output_stale():
    if not paired_json.exists() or not paired_table.exists() or not paired_manifest.exists():
        return True, 'paired output is missing'
    try:
        payload = json.loads(paired_json.read_text(encoding='utf-8'))
        inputs = payload.get('inputs', {})
        expected = {
            'full_predictions': inputs.get('full_predictions', {}).get('sha256'),
            'minirocket_predictions': inputs.get('comparator_predictions', {}).get('sha256'),
            'freeze_manifest': inputs.get('freeze_manifest', {}).get('sha256'),
            'minirocket_summary': inputs.get('minirocket', {}).get('summary', {}).get('sha256'),
            'minirocket_manifest': inputs.get('minirocket', {}).get('manifest', {}).get('sha256'),
        }
        current = {label: sha256_file(path) for label, path in paired_inputs.items()}
    except Exception as exc:
        return True, f'could not validate existing paired output: {exc!r}'
    mismatched = [label for label in expected if expected.get(label) != current.get(label)]
    if mismatched:
        return True, 'input SHA changed for: ' + ', '.join(mismatched)
    return False, 'existing paired output matches current input SHA values'

stale, stale_reason = paired_output_stale()
print('paired output stale:', stale, '|', stale_reason)

paired_command = (
    'python -u scripts/revision/11_paired_full_vs_minirocket.py '
    '--full-predictions reports/revision/predictions/oof_final_ema_predictions.npz '
    '--comparator-predictions reports/revision/predictions/minirocket_only_oof_predictions.npz '
    '--freeze-manifest reports/revision/manifests/oof_final_ema_freeze_manifest.json '
    '--comparator-summary reports/revision/metrics/minirocket_only_baseline_summary.json '
    '--comparator-manifest reports/revision/manifests/minirocket_only_baseline_manifest.json '
    '--threshold 0.5 --n-bins 15 --n-boot 1000 --seed 42 '
    '--expected-records 44186 --expected-classes 27 --expected-checkpoint-kind final_ema '
    '--require-manuscript-ready'
)
print('paired command:', paired_command)

if RUN_PAIRED_FULL_VS_MINIROCKET and missing_paired_inputs:
    raise FileNotFoundError('Paired comparison requires Notebook 02/03/04 artifacts: ' + '; '.join(missing_paired_inputs))
elif RUN_PAIRED_FULL_VS_MINIROCKET and (FORCE_RERUN_PAIRED_COMPARISON or stale):
    run(paired_command, log_path='reports/revision/logs/paired_full_vs_minirocket.log')
    if 'MIRROR_REVISION_ROOT' in globals():
        run(
            f'python -u scripts/revision/artifact_mirror.py publish --mirror-root "{MIRROR_REVISION_ROOT}"',
            log_path='reports/revision/logs/paired_full_vs_minirocket_immediate_mirror_publish.log',
        )
elif paired_json.exists():
    print('Skipping paired comparison because existing output passed input-SHA validation. Set FORCE_RERUN_PAIRED_COMPARISON=True to rerun.')
else:
    print('Paired comparison disabled. Set RUN_PAIRED_FULL_VS_MINIROCKET=True after MiniRocket-only artifacts exist.')

if paired_json.exists():
    payload = json.loads(paired_json.read_text(encoding='utf-8'))
    print('paired status:', payload.get('status'))
    print('paired output:', paired_json)
if paired_table.exists():
    paired_df = pd.read_csv(paired_table)
    display(paired_df[['metric', 'full_value', 'comparator_value', 'improvement_full_over_comparator', 'improvement_ci_low', 'improvement_ci_high', 'interpretation', 'safe_wording']])


## Paired Full vs ResNet1D/CNN Evidence

This cell compares Full ECG-RAMBA against the raw-ECG ResNet1D/CNN baseline with the same record-level paired bootstrap contract used for MiniRocket. It is the key in-domain architecture baseline evidence; interpret results metric-by-metric rather than as global superiority.


In [ ]:
RUN_PAIRED_FULL_VS_RESNET = True
FORCE_RERUN_PAIRED_RESNET_COMPARISON = False

paired_resnet_json = revision_root / 'metrics' / 'paired_full_vs_resnet_comparison.json'
paired_resnet_table = revision_root / 'tables' / 'table_paired_full_vs_resnet.csv'
paired_resnet_samples = revision_root / 'metrics' / 'paired_full_vs_resnet_bootstrap_samples.csv'
paired_resnet_manifest = revision_root / 'manifests' / 'paired_full_vs_resnet_manifest.json'

paired_resnet_inputs = {
    'full_predictions': revision_root / 'predictions' / 'oof_final_ema_predictions.npz',
    'resnet_predictions': revision_root / 'predictions' / 'resnet1d_cnn_oof_predictions.npz',
    'freeze_manifest': revision_root / 'manifests' / 'oof_final_ema_freeze_manifest.json',
    'resnet_summary': revision_root / 'metrics' / 'resnet1d_cnn_baseline_summary.json',
    'resnet_manifest': revision_root / 'manifests' / 'resnet1d_cnn_baseline_manifest.json',
}
for label, path in paired_resnet_inputs.items():
    print(f'{label:24}: exists={path.exists()} size={path.stat().st_size if path.exists() else None} path={path}')
missing_paired_resnet_inputs = [str(path) for path in paired_resnet_inputs.values() if not path.exists()]

from scripts.revision.common import sha256_file

def paired_resnet_output_stale():
    if not paired_resnet_json.exists() or not paired_resnet_table.exists() or not paired_resnet_manifest.exists():
        return True, 'paired ResNet output is missing'
    try:
        payload = json.loads(paired_resnet_json.read_text(encoding='utf-8'))
        inputs = payload.get('inputs', {})
        expected = {
            'full_predictions': inputs.get('full_predictions', {}).get('sha256'),
            'resnet_predictions': inputs.get('comparator_predictions', {}).get('sha256'),
            'freeze_manifest': inputs.get('freeze_manifest', {}).get('sha256'),
            'resnet_summary': inputs.get('resnet', {}).get('summary', {}).get('sha256'),
            'resnet_manifest': inputs.get('resnet', {}).get('manifest', {}).get('sha256'),
        }
        current = {label: sha256_file(path) for label, path in paired_resnet_inputs.items()}
    except Exception as exc:
        return True, f'could not validate existing paired ResNet output: {exc!r}'
    mismatched = [label for label in expected if expected.get(label) != current.get(label)]
    if mismatched:
        return True, 'input SHA changed for: ' + ', '.join(mismatched)
    return False, 'existing paired ResNet output matches current input SHA values'

resnet_stale, resnet_stale_reason = paired_resnet_output_stale()
print('paired ResNet output stale:', resnet_stale, '|', resnet_stale_reason)

paired_resnet_command = (
    'python -u scripts/revision/15_paired_full_vs_resnet.py '
    '--full-predictions reports/revision/predictions/oof_final_ema_predictions.npz '
    '--comparator-predictions reports/revision/predictions/resnet1d_cnn_oof_predictions.npz '
    '--freeze-manifest reports/revision/manifests/oof_final_ema_freeze_manifest.json '
    '--comparator-summary reports/revision/metrics/resnet1d_cnn_baseline_summary.json '
    '--comparator-manifest reports/revision/manifests/resnet1d_cnn_baseline_manifest.json '
    '--threshold 0.5 --n-bins 15 --n-boot 1000 --seed 42 '
    '--expected-records 44186 --expected-classes 27 --expected-checkpoint-kind final_ema '
    '--require-manuscript-ready'
)
print('paired ResNet command:', paired_resnet_command)

if RUN_PAIRED_FULL_VS_RESNET and missing_paired_resnet_inputs:
    raise FileNotFoundError('Paired ResNet comparison requires Notebook 02/03/04 artifacts: ' + '; '.join(missing_paired_resnet_inputs))
elif RUN_PAIRED_FULL_VS_RESNET and (FORCE_RERUN_PAIRED_RESNET_COMPARISON or resnet_stale):
    run(paired_resnet_command, log_path='reports/revision/logs/paired_full_vs_resnet.log')
    if 'MIRROR_REVISION_ROOT' in globals():
        run(
            f'python -u scripts/revision/artifact_mirror.py publish --mirror-root "{MIRROR_REVISION_ROOT}"',
            log_path='reports/revision/logs/paired_full_vs_resnet_immediate_mirror_publish.log',
        )
elif paired_resnet_json.exists():
    print('Skipping paired ResNet comparison because existing output passed input-SHA validation. Set FORCE_RERUN_PAIRED_RESNET_COMPARISON=True to rerun.')
else:
    print('Paired ResNet comparison disabled. Set RUN_PAIRED_FULL_VS_RESNET=True after ResNet1D/CNN artifacts exist.')

if paired_resnet_json.exists():
    payload = json.loads(paired_resnet_json.read_text(encoding='utf-8'))
    print('paired ResNet status:', payload.get('status'))
    print('paired ResNet output:', paired_resnet_json)
if paired_resnet_table.exists():
    paired_resnet_df = pd.read_csv(paired_resnet_table)
    display(paired_resnet_df[['metric', 'full_value', 'comparator_value', 'improvement_full_over_comparator', 'improvement_ci_low', 'improvement_ci_high', 'interpretation', 'safe_wording']])


## Paired Full vs Raw Mamba Evidence

Paired record-level bootstrap comparison. This runs only when Raw Mamba baseline artifacts exist, so ordinary notebook reruns do not fail before the Raw Mamba experiment is complete.


In [ ]:
RUN_PAIRED_FULL_VS_RAW_MAMBA = 'auto'
FORCE_RERUN_PAIRED_RAW_MAMBA_COMPARISON = False

paired_raw_json = revision_root / 'metrics' / 'paired_full_vs_raw_mamba_comparison.json'
paired_raw_table = revision_root / 'tables' / 'table_paired_full_vs_raw_mamba.csv'
paired_raw_samples = revision_root / 'metrics' / 'paired_full_vs_raw_mamba_bootstrap_samples.csv'
paired_raw_manifest = revision_root / 'manifests' / 'paired_full_vs_raw_mamba_manifest.json'

paired_raw_inputs = {
    'full_predictions': revision_root / 'predictions' / 'oof_final_ema_predictions.npz',
    'raw_mamba_predictions': revision_root / 'predictions' / 'raw_mamba_oof_predictions.npz',
    'freeze_manifest': revision_root / 'manifests' / 'oof_final_ema_freeze_manifest.json',
    'raw_mamba_summary': revision_root / 'metrics' / 'raw_mamba_baseline_summary.json',
    'raw_mamba_manifest': revision_root / 'manifests' / 'raw_mamba_baseline_manifest.json',
}
for label, path in paired_raw_inputs.items():
    print(f'{label:24}: exists={path.exists()} size={path.stat().st_size if path.exists() else None} path={path}')
missing_paired_raw_inputs = [str(path) for path in paired_raw_inputs.values() if not path.exists()]

from scripts.revision.common import sha256_file

def paired_raw_output_stale():
    if not paired_raw_json.exists() or not paired_raw_table.exists() or not paired_raw_manifest.exists():
        return True, 'paired Raw Mamba output is missing'
    try:
        payload = json.loads(paired_raw_json.read_text(encoding='utf-8'))
        inputs = payload.get('inputs', {})
        expected = {
            'full_predictions': inputs.get('full_predictions', {}).get('sha256'),
            'raw_mamba_predictions': inputs.get('comparator_predictions', {}).get('sha256'),
            'freeze_manifest': inputs.get('freeze_manifest', {}).get('sha256'),
            'raw_mamba_summary': inputs.get('raw_mamba', {}).get('summary', {}).get('sha256'),
            'raw_mamba_manifest': inputs.get('raw_mamba', {}).get('manifest', {}).get('sha256'),
        }
        current = {label: sha256_file(path) for label, path in paired_raw_inputs.items()}
    except Exception as exc:
        return True, f'could not validate existing paired Raw Mamba output: {exc!r}'
    mismatched = [label for label in expected if expected.get(label) != current.get(label)]
    if mismatched:
        return True, 'input SHA changed for: ' + ', '.join(mismatched)
    return False, 'existing paired Raw Mamba output matches current input SHA values'

raw_stale, raw_stale_reason = paired_raw_output_stale()
print('paired Raw Mamba output stale:', raw_stale, '|', raw_stale_reason)

paired_raw_command = (
    'python -u scripts/revision/17_paired_full_vs_raw_mamba.py '
    '--full-predictions reports/revision/predictions/oof_final_ema_predictions.npz '
    '--comparator-predictions reports/revision/predictions/raw_mamba_oof_predictions.npz '
    '--freeze-manifest reports/revision/manifests/oof_final_ema_freeze_manifest.json '
    '--comparator-summary reports/revision/metrics/raw_mamba_baseline_summary.json '
    '--comparator-manifest reports/revision/manifests/raw_mamba_baseline_manifest.json '
    '--threshold 0.5 --n-bins 15 --n-boot 1000 --seed 42 '
    '--expected-records 44186 --expected-classes 27 --expected-checkpoint-kind final_ema '
    '--require-manuscript-ready'
)
print('paired Raw Mamba command:', paired_raw_command)

auto_run_raw_pair = RUN_PAIRED_FULL_VS_RAW_MAMBA == 'auto' and not missing_paired_raw_inputs
explicit_run_raw_pair = RUN_PAIRED_FULL_VS_RAW_MAMBA is True
should_run_raw_pair = auto_run_raw_pair or explicit_run_raw_pair

if explicit_run_raw_pair and missing_paired_raw_inputs:
    raise FileNotFoundError('Paired Raw Mamba comparison requires Raw Mamba artifacts: ' + '; '.join(missing_paired_raw_inputs))
elif should_run_raw_pair and (FORCE_RERUN_PAIRED_RAW_MAMBA_COMPARISON or raw_stale):
    run(paired_raw_command, log_path='reports/revision/logs/paired_full_vs_raw_mamba.log')
    if 'MIRROR_REVISION_ROOT' in globals():
        run(
            f'python -u scripts/revision/artifact_mirror.py publish --mirror-root "{MIRROR_REVISION_ROOT}"',
            log_path='reports/revision/logs/paired_full_vs_raw_mamba_immediate_mirror_publish.log',
        )
elif paired_raw_json.exists():
    print('Skipping paired Raw Mamba comparison because existing output passed input-SHA validation. Set FORCE_RERUN_PAIRED_RAW_MAMBA_COMPARISON=True to rerun.')
else:
    print('Paired Raw Mamba comparison skipped because Raw Mamba artifacts are not complete yet. Run the Raw Mamba baseline cell first.')

if paired_raw_json.exists():
    payload = json.loads(paired_raw_json.read_text(encoding='utf-8'))
    print('paired Raw Mamba status:', payload.get('status'))
    print('paired Raw Mamba output:', paired_raw_json)
if paired_raw_table.exists():
    paired_raw_df = pd.read_csv(paired_raw_table)
    display(paired_raw_df[['metric', 'full_value', 'comparator_value', 'improvement_full_over_comparator', 'improvement_ci_low', 'improvement_ci_high', 'interpretation', 'safe_wording']])


## Paired Full vs Transformer ECG Evidence

Optional paired bootstrap comparison for the Transformer ECG comparator. Runs only when Transformer artifacts exist or when explicitly enabled.


In [ ]:
RUN_PAIRED_FULL_VS_TRANSFORMER = 'auto'
FORCE_RERUN_PAIRED_TRANSFORMER_COMPARISON = False

paired_transformer_json = revision_root / 'metrics' / 'paired_full_vs_transformer_comparison.json'
paired_transformer_table = revision_root / 'tables' / 'table_paired_full_vs_transformer.csv'
paired_transformer_samples = revision_root / 'metrics' / 'paired_full_vs_transformer_bootstrap_samples.csv'
paired_transformer_manifest = revision_root / 'manifests' / 'paired_full_vs_transformer_manifest.json'

paired_transformer_inputs = {
    'full_predictions': revision_root / 'predictions' / 'oof_final_ema_predictions.npz',
    'transformer_predictions': revision_root / 'predictions' / 'transformer_ecg_oof_predictions.npz',
    'freeze_manifest': revision_root / 'manifests' / 'oof_final_ema_freeze_manifest.json',
    'transformer_summary': revision_root / 'metrics' / 'transformer_ecg_baseline_summary.json',
    'transformer_manifest': revision_root / 'manifests' / 'transformer_ecg_baseline_manifest.json',
}
for label, path in paired_transformer_inputs.items():
    print(f'{label:24}: exists={path.exists()} size={path.stat().st_size if path.exists() else None} path={path}')
missing_paired_transformer_inputs = [str(path) for path in paired_transformer_inputs.values() if not path.exists()]

from scripts.revision.common import sha256_file

def paired_transformer_output_stale():
    if not paired_transformer_json.exists() or not paired_transformer_table.exists() or not paired_transformer_manifest.exists():
        return True, 'paired Transformer output is missing'
    try:
        payload = json.loads(paired_transformer_json.read_text(encoding='utf-8'))
        inputs = payload.get('inputs', {})
        expected = {
            'full_predictions': inputs.get('full_predictions', {}).get('sha256'),
            'transformer_predictions': inputs.get('comparator_predictions', {}).get('sha256'),
            'freeze_manifest': inputs.get('freeze_manifest', {}).get('sha256'),
            'transformer_summary': inputs.get('transformer', {}).get('summary', {}).get('sha256'),
            'transformer_manifest': inputs.get('transformer', {}).get('manifest', {}).get('sha256'),
        }
        current = {label: sha256_file(path) for label, path in paired_transformer_inputs.items()}
    except Exception as exc:
        return True, f'could not validate existing paired Transformer output: {exc!r}'
    mismatched = [label for label in expected if expected.get(label) != current.get(label)]
    if mismatched:
        return True, 'input SHA changed for: ' + ', '.join(mismatched)
    return False, 'existing paired Transformer output matches current input SHA values'

transformer_stale, transformer_stale_reason = paired_transformer_output_stale()
print('paired Transformer output stale:', transformer_stale, '|', transformer_stale_reason)

paired_transformer_command = (
    'python -u scripts/revision/25_paired_full_vs_transformer.py '
    '--full-predictions reports/revision/predictions/oof_final_ema_predictions.npz '
    '--comparator-predictions reports/revision/predictions/transformer_ecg_oof_predictions.npz '
    '--freeze-manifest reports/revision/manifests/oof_final_ema_freeze_manifest.json '
    '--comparator-summary reports/revision/metrics/transformer_ecg_baseline_summary.json '
    '--comparator-manifest reports/revision/manifests/transformer_ecg_baseline_manifest.json '
    '--threshold 0.5 --n-bins 15 --n-boot 1000 --seed 42 '
    '--expected-records 44186 --expected-classes 27 --expected-checkpoint-kind final_ema '
    '--require-manuscript-ready'
)
print('paired Transformer command:', paired_transformer_command)

auto_run_transformer_pair = RUN_PAIRED_FULL_VS_TRANSFORMER == 'auto' and not missing_paired_transformer_inputs
explicit_run_transformer_pair = RUN_PAIRED_FULL_VS_TRANSFORMER is True
should_run_transformer_pair = auto_run_transformer_pair or explicit_run_transformer_pair

if explicit_run_transformer_pair and missing_paired_transformer_inputs:
    raise FileNotFoundError('Paired Transformer comparison requires Transformer ECG artifacts: ' + '; '.join(missing_paired_transformer_inputs))
elif should_run_transformer_pair and (FORCE_RERUN_PAIRED_TRANSFORMER_COMPARISON or transformer_stale):
    run(paired_transformer_command, log_path='reports/revision/logs/paired_full_vs_transformer.log')
    if 'MIRROR_REVISION_ROOT' in globals():
        run(
            f'python -u scripts/revision/artifact_mirror.py publish --mirror-root "{MIRROR_REVISION_ROOT}"',
            log_path='reports/revision/logs/paired_full_vs_transformer_immediate_mirror_publish.log',
        )
elif paired_transformer_json.exists():
    print('Skipping paired Transformer comparison because existing output passed input-SHA validation. Set FORCE_RERUN_PAIRED_TRANSFORMER_COMPARISON=True to rerun.')
else:
    print('Paired Transformer comparison skipped because Transformer ECG artifacts are not complete yet. Run the optional Transformer baseline cell first if needed.')

if paired_transformer_json.exists():
    payload = json.loads(paired_transformer_json.read_text(encoding='utf-8'))
    print('paired Transformer status:', payload.get('status'))
    print('paired Transformer output:', paired_transformer_json)
if paired_transformer_table.exists():
    paired_transformer_df = pd.read_csv(paired_transformer_table)
    display(paired_transformer_df[['metric', 'full_value', 'comparator_value', 'improvement_full_over_comparator', 'improvement_ci_low', 'improvement_ci_high', 'interpretation', 'safe_wording']])


## Paired Full vs Hybrid MiniRocket Morphology MLP Evidence

Optional paired bootstrap comparison for the learnable MiniRocket morphology-head sensitivity baseline. Runs only when Hybrid morphology artifacts exist or when explicitly enabled.


In [ ]:
RUN_PAIRED_FULL_VS_HYBRID_MORPHOLOGY = 'auto'
FORCE_RERUN_PAIRED_HYBRID_MORPHOLOGY_COMPARISON = False

paired_hybrid_json = revision_root / 'metrics' / 'paired_full_vs_hybrid_morphology_comparison.json'
paired_hybrid_table = revision_root / 'tables' / 'table_paired_full_vs_hybrid_morphology.csv'
paired_hybrid_samples = revision_root / 'metrics' / 'paired_full_vs_hybrid_morphology_bootstrap_samples.csv'
paired_hybrid_manifest = revision_root / 'manifests' / 'paired_full_vs_hybrid_morphology_manifest.json'

paired_hybrid_inputs = {
    'full_predictions': revision_root / 'predictions' / 'oof_final_ema_predictions.npz',
    'hybrid_predictions': revision_root / 'predictions' / 'hybrid_morphology_oof_predictions.npz',
    'freeze_manifest': revision_root / 'manifests' / 'oof_final_ema_freeze_manifest.json',
    'hybrid_summary': revision_root / 'metrics' / 'hybrid_morphology_baseline_summary.json',
    'hybrid_manifest': revision_root / 'manifests' / 'hybrid_morphology_baseline_manifest.json',
}
for label, path in paired_hybrid_inputs.items():
    print(f'{label:24}: exists={path.exists()} size={path.stat().st_size if path.exists() else None} path={path}')
missing_paired_hybrid_inputs = [str(path) for path in paired_hybrid_inputs.values() if not path.exists()]

from scripts.revision.common import sha256_file

def paired_hybrid_output_stale():
    if not paired_hybrid_json.exists() or not paired_hybrid_table.exists() or not paired_hybrid_manifest.exists():
        return True, 'paired Hybrid morphology output is missing'
    try:
        payload = json.loads(paired_hybrid_json.read_text(encoding='utf-8'))
        inputs = payload.get('inputs', {})
        expected = {
            'full_predictions': inputs.get('full_predictions', {}).get('sha256'),
            'hybrid_predictions': inputs.get('comparator_predictions', {}).get('sha256'),
            'freeze_manifest': inputs.get('freeze_manifest', {}).get('sha256'),
            'hybrid_summary': inputs.get('hybrid_morphology', {}).get('summary', {}).get('sha256'),
            'hybrid_manifest': inputs.get('hybrid_morphology', {}).get('manifest', {}).get('sha256'),
        }
        current = {label: sha256_file(path) for label, path in paired_hybrid_inputs.items()}
    except Exception as exc:
        return True, f'could not validate existing paired Hybrid morphology output: {exc!r}'
    mismatched = [label for label in expected if expected.get(label) != current.get(label)]
    if mismatched:
        return True, 'input SHA changed for: ' + ', '.join(mismatched)
    return False, 'existing paired Hybrid morphology output matches current input SHA values'

hybrid_stale, hybrid_stale_reason = paired_hybrid_output_stale()
print('paired Hybrid morphology output stale:', hybrid_stale, '|', hybrid_stale_reason)

paired_hybrid_command = (
    'python -u scripts/revision/27_paired_full_vs_hybrid_morphology.py '
    '--full-predictions reports/revision/predictions/oof_final_ema_predictions.npz '
    '--comparator-predictions reports/revision/predictions/hybrid_morphology_oof_predictions.npz '
    '--freeze-manifest reports/revision/manifests/oof_final_ema_freeze_manifest.json '
    '--comparator-summary reports/revision/metrics/hybrid_morphology_baseline_summary.json '
    '--comparator-manifest reports/revision/manifests/hybrid_morphology_baseline_manifest.json '
    '--threshold 0.5 --n-bins 15 --n-boot 1000 --seed 42 '
    '--expected-records 44186 --expected-classes 27 --expected-checkpoint-kind final_ema '
    '--require-manuscript-ready'
)
print('paired Hybrid morphology command:', paired_hybrid_command)

auto_run_hybrid_pair = RUN_PAIRED_FULL_VS_HYBRID_MORPHOLOGY == 'auto' and not missing_paired_hybrid_inputs
explicit_run_hybrid_pair = RUN_PAIRED_FULL_VS_HYBRID_MORPHOLOGY is True
should_run_hybrid_pair = auto_run_hybrid_pair or explicit_run_hybrid_pair
if explicit_run_hybrid_pair and missing_paired_hybrid_inputs:
    raise FileNotFoundError('Paired Hybrid morphology comparison requires artifacts: ' + '; '.join(missing_paired_hybrid_inputs))
elif should_run_hybrid_pair and (FORCE_RERUN_PAIRED_HYBRID_MORPHOLOGY_COMPARISON or hybrid_stale):
    run(paired_hybrid_command, log_path='reports/revision/logs/paired_full_vs_hybrid_morphology.log')
    if 'MIRROR_REVISION_ROOT' in globals():
        run(
            f'python -u scripts/revision/artifact_mirror.py publish --mirror-root "{MIRROR_REVISION_ROOT}"',
            log_path='reports/revision/logs/paired_full_vs_hybrid_morphology_mirror_publish.log',
        )
elif should_run_hybrid_pair:
    print('Skipping paired Hybrid morphology comparison because existing output passed input-SHA validation. Set FORCE_RERUN_PAIRED_HYBRID_MORPHOLOGY_COMPARISON=True to rerun.')
else:
    print('Paired Hybrid morphology comparison not run. Auto mode waits for Hybrid morphology baseline artifacts.')

for path in [paired_hybrid_json, paired_hybrid_table, paired_hybrid_samples, paired_hybrid_manifest]:
    print(f'{path}: exists={path.exists()} size={path.stat().st_size if path.exists() else None}')


## Component And Claim Evidence Ledger

This ledger connects the current artifacts to the claims that can and cannot be made after notebooks 02 and 03. It is designed for rebuttal traceability.


In [ ]:
pooling_sensitivity_path = revision_root / 'metrics' / 'pooling_sensitivity.csv'
pooling_decision_path = revision_root / 'metrics' / 'pooling_decision_summary.json'

paired_paths = {
    'MiniRocket-only': {
        'json': revision_root / 'metrics' / 'paired_full_vs_minirocket_comparison.json',
        'table': revision_root / 'tables' / 'table_paired_full_vs_minirocket.csv',
    },
    'ResNet1D/CNN': {
        'json': revision_root / 'metrics' / 'paired_full_vs_resnet_comparison.json',
        'table': revision_root / 'tables' / 'table_paired_full_vs_resnet.csv',
    },
    'Raw Mamba': {
        'json': revision_root / 'metrics' / 'paired_full_vs_raw_mamba_comparison.json',
        'table': revision_root / 'tables' / 'table_paired_full_vs_raw_mamba.csv',
    },
    'Transformer ECG': {
        'json': revision_root / 'metrics' / 'paired_full_vs_transformer_comparison.json',
        'table': revision_root / 'tables' / 'table_paired_full_vs_transformer.csv',
    },
    'Hybrid MiniRocket-MLP': {
        'json': revision_root / 'metrics' / 'paired_full_vs_hybrid_morphology_comparison.json',
        'table': revision_root / 'tables' / 'table_paired_full_vs_hybrid_morphology.csv',
    },
}
paired_payloads = {}
paired_artifact_paths = []
for label, paths in paired_paths.items():
    available = paths['json'].exists() and paths['table'].exists()
    print(f'paired {label:15}: available={available} json={paths["json"]}')
    if available:
        paired_payloads[label] = json.loads(paths['json'].read_text(encoding='utf-8'))
        paired_artifact_paths.extend([str(paths['json']), str(paths['table'])])
    else:
        paired_payloads[label] = None
        paired_artifact_paths.append(str(paths['json']))
paired_available = any(payload is not None for payload in paired_payloads.values())
paired_artifacts = '; '.join([str(revision_root / 'metrics' / 'baseline_summary.csv')] + paired_artifact_paths)

def paired_brief(label, metrics=('pr_auc_macro', 'roc_auc_macro', 'f1_macro', 'brier_macro', 'ece_macro')):
    payload = paired_payloads.get(label)
    if not isinstance(payload, dict):
        return f'{label}: paired comparison unavailable'
    metric_payload = payload.get('metrics', {})
    parts = []
    for metric in metrics:
        row = metric_payload.get(metric, {})
        interpretation = row.get('interpretation', 'missing')
        delta = row.get('improvement_full_over_comparator')
        if isinstance(delta, (int, float)):
            parts.append(f'{metric}={interpretation} (Full improvement={delta:.4f})')
        else:
            parts.append(f'{metric}={interpretation}')
    return f'{label}: ' + ', '.join(parts)

paired_guidance = ' '.join([
    paired_brief('MiniRocket-only'),
    paired_brief('ResNet1D/CNN'),
    paired_brief('Raw Mamba'),
    paired_brief('Transformer ECG'),
    paired_brief('Hybrid MiniRocket-MLP'),
])

if pooling_sensitivity_path.exists() and pooling_decision_path.exists():
    pooling_decision = json.loads(pooling_decision_path.read_text(encoding='utf-8'))
    c05_status = 'supported_by_pooling_sensitivity_with_tradeoff_wording'
    c05_artifacts = '; '.join([str(pooling_sensitivity_path), str(pooling_decision_path)])
    c05_wording = pooling_decision.get('safe_wording', 'Report pooling sensitivity table and avoid optimality wording unless supported.')
else:
    pooling_decision = None
    c05_status = 'pending_pooling_sensitivity'
    c05_artifacts = str(pooling_sensitivity_path)
    c05_wording = 'Run notebook 06 pooling sensitivity before justifying Q=3 beyond the frozen OOF contract.'

required_fair_baselines = ['Raw Mamba', 'ResNet1D/CNN', 'MiniRocket-only', 'HRV-only']
complete_statuses = {'complete_feature_baseline_from_notebook05', 'complete_feature_baseline_from_script10', 'complete_architecture_baseline_from_script14', 'complete_architecture_baseline_from_script16', 'complete_frozen_oof'}
baseline_status = dict(zip(baseline_df['baseline_name'], baseline_df['status']))
missing_fair_baselines = [
    name for name in required_fair_baselines
    if baseline_status.get(name) not in complete_statuses
]
if missing_fair_baselines:
    c01_status = 'blocked_fair_baselines_missing_with_paired_comparators_available' if paired_available else 'blocked_fair_baselines_missing'
    c01_wording = 'Do not claim broad fair-baseline superiority until these rows are complete: ' + ', '.join(missing_fair_baselines) + '.'
    if paired_available:
        c01_wording += ' Use completed paired comparator tables only as metric-specific evidence. ' + paired_guidance
else:
    c01_status = 'fair_baseline_rows_complete_with_paired_comparisons' if paired_available else 'fair_baseline_rows_complete_pending_statistical_comparison'
    c01_wording = ('Only claim superiority for metrics whose paired bootstrap delta supports it. ' + paired_guidance) if paired_available else 'Fair baseline rows are complete; only claim superiority after paired uncertainty/delta analysis supports it.'

component_rows = [
    {
        'claim_id': 'C01',
        'claim_topic': 'fair baseline superiority',
        'current_status': c01_status,
        'supporting_artifacts': paired_artifacts,
        'safe_wording': c01_wording,
    },
    {
        'claim_id': 'C02',
        'claim_topic': 'fixed-threshold ranking-decision gap',
        'current_status': 'supported_by_calibration_artifacts_with_limitations',
        'supporting_artifacts': '; '.join([
            str(calibration_path),
            str(revision_root / 'figures' / 'reliability_oof_final_ema_predictions.png'),
            str(revision_root / 'tables' / 'reliability_bins_oof_final_ema_predictions.csv'),
            *paired_artifact_paths,
        ]),
        'safe_wording': 'Frozen OOF calibration and paired comparator tables show metric-specific tradeoffs; avoid global superiority or clinical readiness wording.',
    },
    {
        'claim_id': 'C05',
        'claim_topic': 'Q=3 pooling operating point',
        'current_status': c05_status,
        'supporting_artifacts': c05_artifacts,
        'safe_wording': c05_wording,
    },
    {
        'claim_id': 'C06',
        'claim_topic': 'protocol-faithful OOF evaluation',
        'current_status': 'partial_oof_supported',
        'supporting_artifacts': '; '.join([str(freeze_path), str(run_manifest_path), str(calibration_path)]),
        'safe_wording': 'OOF protocol is frozen and traceable; external outputs remain experimental until separate readiness gates pass.',
    },
]
component_df = pd.DataFrame(component_rows)
component_table = revision_root / 'tables' / 'table_component_checks.csv'
component_json = revision_root / 'metrics' / 'component_check_summary.json'
component_df.to_csv(component_table, index=False)
save_json(component_json, {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'git_commit': git_commit(),
    'source_freeze_manifest': str(freeze_path),
    'source_calibration': str(calibration_path),
    'source_pooling_decision': str(pooling_decision_path) if pooling_decision_path.exists() else None,
    'source_paired_minirocket_comparison': str(paired_paths['MiniRocket-only']['json']) if paired_paths['MiniRocket-only']['json'].exists() else None,
    'source_paired_resnet_comparison': str(paired_paths['ResNet1D/CNN']['json']) if paired_paths['ResNet1D/CNN']['json'].exists() else None,
    'source_paired_raw_mamba_comparison': str(paired_paths['Raw Mamba']['json']) if paired_paths['Raw Mamba']['json'].exists() else None,
    'source_paired_transformer_comparison': str(paired_paths['Transformer ECG']['json']) if paired_paths['Transformer ECG']['json'].exists() else None,
    'source_paired_hybrid_morphology_comparison': str(paired_paths['Hybrid MiniRocket-MLP']['json']) if paired_paths['Hybrid MiniRocket-MLP']['json'].exists() else None,
    'source_paired_minirocket_table': str(paired_paths['MiniRocket-only']['table']) if paired_paths['MiniRocket-only']['table'].exists() else None,
    'source_paired_resnet_table': str(paired_paths['ResNet1D/CNN']['table']) if paired_paths['ResNet1D/CNN']['table'].exists() else None,
    'source_paired_raw_mamba_table': str(paired_paths['Raw Mamba']['table']) if paired_paths['Raw Mamba']['table'].exists() else None,
    'source_paired_transformer_table': str(paired_paths['Transformer ECG']['table']) if paired_paths['Transformer ECG']['table'].exists() else None,
    'source_paired_hybrid_morphology_table': str(paired_paths['Hybrid MiniRocket-MLP']['table']) if paired_paths['Hybrid MiniRocket-MLP']['table'].exists() else None,
    'pooling_decision': pooling_decision,
    'paired_comparisons': paired_payloads,
    'paired_guidance': paired_guidance,
    'claims': component_rows,
})
print('Wrote:', component_table)
print('Wrote:', component_json)
display(component_df)


## Heavy Baseline Guard

The remaining model-specific fair baseline runners are not implemented in this branch. This cell records their intended command surface and stops accidental placeholder execution.


In [ ]:
planned = {}
print('MiniRocket-only, ResNet1D/CNN, and Raw Mamba now have dedicated runner cells earlier in this notebook.')
print('HRV-only is intentionally excluded from this heavy-runner guard; Notebook 04 only imports its canonical artifact from Notebook 05.')

if RUN_HEAVY_BASELINES and planned:
    for name, command in planned.items():
        run(command, log_path=f'reports/revision/logs/baseline_{name.lower().replace("/", "_").replace(" ", "_")}.log')
elif RUN_HEAVY_BASELINES:
    print('No TBD heavy baseline runners remain in this guard. Use the dedicated runner cells above.')
else:
    print('Heavy baseline guard complete. Use dedicated cells for MiniRocket-only, ResNet1D/CNN, and Raw Mamba as needed.')

## Inventory And Stable Mirror

After notebook 04 writes evidence tables, freeze the revision artifact inventory and publish the artifacts to the Drive mirror for reuse.


In [ ]:
import inspect
if 'log_path' not in inspect.signature(run).parameters:
    raise RuntimeError('Notebook command helper run() was overwritten. Rerun the Setup cell before this inventory/mirror cell.')

run(
    'python -u scripts/revision/05_artifact_inventory.py',
    log_path='reports/revision/logs/baseline_component_artifact_inventory.log',
)

mirror_root = DRIVE_ROOT / 'revision_artifacts' / 'reports' / 'revision'
run(
    f'python -u scripts/revision/artifact_mirror.py publish --mirror-root "{mirror_root}"',
    log_path='reports/revision/logs/baseline_component_mirror_publish.log',
)


## Output Summary


In [ ]:
expected_outputs = [
    revision_root / 'manifests' / 'baseline_component_input_contract.json',
    revision_root / 'manifests' / 'baseline_component_plan_alignment.json',
    revision_root / 'metrics' / 'no_skill_reference_baselines.csv',
    revision_root / 'tables' / 'table_no_skill_reference_baselines.csv',
    revision_root / 'metrics' / 'baseline_summary.csv',
    revision_root / 'tables' / 'table_baselines.csv',
    revision_root / 'metrics' / 'component_check_summary.json',
    revision_root / 'tables' / 'table_component_checks.csv',
    revision_root / 'manifests' / 'artifacts_manifest.json',
    revision_root / 'manifests' / 'artifacts_manifest.csv',
]
for path in expected_outputs:
    print(path, 'exists=', path.exists(), 'size=', path.stat().st_size if path.exists() else None)

optional_feature_outputs = [
    revision_root / 'metrics' / 'minirocket_only_baseline_summary.json',
    revision_root / 'predictions' / 'minirocket_only_oof_predictions.npz',
    revision_root / 'tables' / 'table_minirocket_only_class_metrics.csv',
    revision_root / 'tables' / 'table_minirocket_only_fold_summary.csv',
    revision_root / 'manifests' / 'minirocket_only_baseline_manifest.json',
    revision_root / 'metrics' / 'resnet1d_cnn_baseline_summary.json',
    revision_root / 'predictions' / 'resnet1d_cnn_oof_predictions.npz',
    revision_root / 'predictions' / 'resnet1d_cnn_slice_predictions.npz',
    revision_root / 'tables' / 'table_resnet1d_cnn_class_metrics.csv',
    revision_root / 'tables' / 'table_resnet1d_cnn_fold_summary.csv',
    revision_root / 'manifests' / 'resnet1d_cnn_baseline_manifest.json',
    *[revision_root / 'predictions' / 'folds' / f'resnet1d_cnn_fold{i}_predictions.npz' for i in range(1, 6)],
    *[revision_root / 'experimental' / 'resnet1d_cnn_checkpoints' / f'fold{i}_resnet1d_cnn_final.pt' for i in range(1, 6)],
    revision_root / 'metrics' / 'raw_mamba_baseline_summary.json',
    revision_root / 'metrics' / 'raw_mamba_fold_cache_status.json',
    revision_root / 'predictions' / 'raw_mamba_oof_predictions.npz',
    revision_root / 'predictions' / 'raw_mamba_slice_predictions.npz',
    revision_root / 'tables' / 'table_raw_mamba_class_metrics.csv',
    revision_root / 'tables' / 'table_raw_mamba_fold_summary.csv',
    revision_root / 'tables' / 'table_raw_mamba_fold_cache_status.csv',
    revision_root / 'manifests' / 'raw_mamba_baseline_manifest.json',
    *[revision_root / 'predictions' / 'folds' / f'raw_mamba_fold{i}_predictions.npz' for i in range(1, 6)],
    *[revision_root / 'experimental' / 'raw_mamba_checkpoints' / f'fold{i}_raw_mamba_final_ema.pt' for i in range(1, 6)],
    revision_root / 'metrics' / 'transformer_ecg_baseline_summary.json',
    revision_root / 'predictions' / 'transformer_ecg_oof_predictions.npz',
    revision_root / 'predictions' / 'transformer_ecg_slice_predictions.npz',
    revision_root / 'tables' / 'table_transformer_ecg_class_metrics.csv',
    revision_root / 'tables' / 'table_transformer_ecg_fold_summary.csv',
    revision_root / 'manifests' / 'transformer_ecg_baseline_manifest.json',
    *[revision_root / 'predictions' / 'folds' / f'transformer_ecg_fold{i}_predictions.npz' for i in range(1, 6)],
    *[revision_root / 'experimental' / 'transformer_ecg_checkpoints' / f'fold{i}_transformer_ecg_final.pt' for i in range(1, 6)],
    revision_root / 'metrics' / 'hybrid_morphology_baseline_summary.json',
    revision_root / 'predictions' / 'hybrid_morphology_oof_predictions.npz',
    revision_root / 'tables' / 'table_hybrid_morphology_class_metrics.csv',
    revision_root / 'tables' / 'table_hybrid_morphology_fold_summary.csv',
    revision_root / 'manifests' / 'hybrid_morphology_baseline_manifest.json',
    revision_root / 'metrics' / 'paired_full_vs_minirocket_comparison.json',
    revision_root / 'metrics' / 'paired_full_vs_minirocket_bootstrap_samples.csv',
    revision_root / 'tables' / 'table_paired_full_vs_minirocket.csv',
    revision_root / 'manifests' / 'paired_full_vs_minirocket_manifest.json',
    revision_root / 'metrics' / 'paired_full_vs_resnet_comparison.json',
    revision_root / 'metrics' / 'paired_full_vs_resnet_bootstrap_samples.csv',
    revision_root / 'tables' / 'table_paired_full_vs_resnet.csv',
    revision_root / 'manifests' / 'paired_full_vs_resnet_manifest.json',
    revision_root / 'metrics' / 'paired_full_vs_raw_mamba_comparison.json',
    revision_root / 'metrics' / 'paired_full_vs_raw_mamba_bootstrap_samples.csv',
    revision_root / 'tables' / 'table_paired_full_vs_raw_mamba.csv',
    revision_root / 'manifests' / 'paired_full_vs_raw_mamba_manifest.json',
    revision_root / 'metrics' / 'paired_full_vs_transformer_comparison.json',
    revision_root / 'metrics' / 'paired_full_vs_transformer_bootstrap_samples.csv',
    revision_root / 'tables' / 'table_paired_full_vs_transformer.csv',
    revision_root / 'manifests' / 'paired_full_vs_transformer_manifest.json',
    revision_root / 'metrics' / 'paired_full_vs_hybrid_morphology_comparison.json',
    revision_root / 'metrics' / 'paired_full_vs_hybrid_morphology_bootstrap_samples.csv',
    revision_root / 'tables' / 'table_paired_full_vs_hybrid_morphology.csv',
    revision_root / 'manifests' / 'paired_full_vs_hybrid_morphology_manifest.json',
]
print('\nOptional feature-baseline outputs:')
for path in optional_feature_outputs:
    print(path, 'exists=', path.exists(), 'size=', path.stat().st_size if path.exists() else None)

print('\nNotebook 04 complete. Use baseline_summary.csv and paired comparator tables as metric-specific evidence; do not claim broad fair-baseline superiority unless every required comparator and paired delta supports it.')
